# SAMS Vision: worker pose (keypoint) training

This notebook trains the pose model that SAMS uses to place body joints on each worker. The live service (`monitoring/services/vision_inference.py`) loads `models/vision/pose/best.pt` and reads its settings from `models/vision/pose/metadata.json`. This notebook writes both files. If they are missing, the service falls back to the stock `models/vision/yolov8n-pose.pt`.

## What a pose model is

A pose model finds each person and 17 body joints (keypoints) on them: nose, eyes, ears, shoulders, elbows, wrists, hips, knees, and ankles. Every joint has a position and a confidence between 0 and 1. The layout and order come from the COCO keypoints dataset.

## What the model has to do in SAMS

The pose model does not decide who is in the frame. The PPE model's `Person` class does that. For every worker box, the service pads the box by 12 percent, crops it (`_crop_worker`), and runs pose on that crop only (`_joints_inside`). The joints are then used in three places:

| Used by | What it needs from the joints |
|---------|-------------------------------|
| `vision_inference._joints_inside` | A pose with no head joint and no shoulder above 0.3 confidence is a false person and the worker is dropped. If pose finds nobody, the worker is kept without joints |
| `ppe_inference.body_regions` | Head (joints 0 to 4), torso (shoulders 5, 6 and hips 11, 12), hands (wrists 9, 10 and elbows 7, 8), and feet (ankles 15, 16). Gear only counts inside these regions |
| `distance_estimation` | Stature from the head (nose, or both eyes) to both ankles, plus pose features for the distance network |

A joint below 0.3 confidence (`_KP_MIN`) is treated as missing everywhere. The joint order is fixed by index in the service code, so the trained model must keep the 17 COCO keypoints in COCO order.

## Why train it at all

The stock `yolov8n-pose.pt` was trained on whole COCO scenes, where people are often small. In SAMS it only ever sees a tight crop around one worker, which is a different kind of picture. This notebook builds training data the same way the service builds its input, scores every model the way the service uses it, and keeps the stock model if fine-tuning does not beat it.

## Plan

| Section | What happens |
|---------|--------------|
| 0 | Setup, configuration, keypoint layout |
| 1 | Download COCO 2017 val images and the Ultralytics pose labels |
| 2 | Cut one padded crop per person, exactly like `_crop_worker` |
| 3 | Train, val, and test splits by source photo |
| 4 | Exploratory data analysis |
| 5 | PCA of body shapes |
| 6 | Training sets, including camera-damage copies |
| 7 | Training setup and the service-matching score |
| 8 | Experiments E0 to E5 |
| 9 | Comparison, choice, and seed repeats |
| 10 | Detailed evaluation on val, including robustness |
| 11 | Final test, metadata for the live service, export check, model card |
| 12 | Limitations and next steps |

## 0. Setup

### 0.1 Paths

The notebook runs in Google Colab or locally. In Colab, data is unpacked on the fast local disk (`/content/SAMS`) and models and results are saved to Google Drive so they survive a disconnect. Locally, the project root is used, so the weights land where `vision_inference.py` looks for them.

In [ ]:
from pathlib import Path

IN_COLAB = False
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print("Running in Colab:", IN_COLAB)

if IN_COLAB:
    SAMS_ROOT = Path("/content/SAMS")
else:
    SAMS_ROOT = Path("./SAMS_runtime")

for folder in [
    SAMS_ROOT / "datasets" / "vision" / "pose",
    SAMS_ROOT / "models" / "vision" / "pose",
    SAMS_ROOT / "results" / "vision" / "pose",
]:
    folder.mkdir(parents=True, exist_ok=True)

print("SAMS_ROOT:", SAMS_ROOT.resolve())

In [ ]:
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_ROOT = Path("/content/drive/MyDrive/SAMS")
else:
    here = Path.cwd().resolve()
    if (here / "manage.py").exists():
        DRIVE_ROOT = here
    elif (here.parent / "manage.py").exists():
        DRIVE_ROOT = here.parent
    else:
        DRIVE_ROOT = Path("./SAMS_drive_mirror")

MODELS_DIR = DRIVE_ROOT / "models" / "vision" / "pose"
RESULTS_DIR = DRIVE_ROOT / "results" / "vision" / "pose"
DATA_DIR = SAMS_ROOT / "datasets" / "vision" / "pose"
PPE_METADATA = DRIVE_ROOT / "models" / "vision" / "ppe" / "metadata.json"

for folder in [MODELS_DIR, RESULTS_DIR, DATA_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("MODELS_DIR: ", MODELS_DIR)
print("RESULTS_DIR:", RESULTS_DIR)
print("DATA_DIR:   ", DATA_DIR)

### 0.2 Packages

`albumentations` 2.x is required because the camera-damage steps use its newer argument names (`std_range`, `quality_range`, `scale_range`). `onnx` and `onnxruntime` are only used for the export check in section 11.

In [ ]:
import subprocess
import sys

PACKAGES = [
    "ultralytics>=8.3.0",
    "albumentations>=2.0.0",
    "scikit-learn",
    "seaborn",
    "pandas",
    "matplotlib",
    "pyyaml",
    "Pillow",
    "onnx",
    "onnxruntime",
]

if IN_COLAB:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *PACKAGES])
    print("Packages installed.")
else:
    print("Not in Colab. Install these packages in your environment if an import fails:")
    print(PACKAGES)

In [ ]:
import gc
import hashlib
import json
import random
import shutil
import time
import urllib.request
import warnings
import zipfile
from collections import Counter
from datetime import datetime, timezone

import albumentations as A
import matplotlib.patches as patches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import sklearn
import torch
import ultralytics
import yaml
from IPython.display import display
from PIL import Image
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from ultralytics import YOLO

warnings.filterwarnings("ignore", category=UserWarning)
sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

version_table = pd.DataFrame([
    {"package": "ultralytics", "version": ultralytics.__version__},
    {"package": "torch", "version": torch.__version__},
    {"package": "albumentations", "version": A.__version__},
    {"package": "scikit-learn", "version": sklearn.__version__},
    {"package": "pandas", "version": pd.__version__},
    {"package": "numpy", "version": np.__version__},
])
display(version_table)

### 0.3 Configuration

Every setting that changes the result is in this one cell, with its meaning in the table below. Change a value here and rerun the notebook.

`LIVE_CONF` is read from the PPE model's `metadata.json`, because the service passes the PPE decision threshold to the pose model (`_run_pose_frame`). `KP_MIN_CONF` is the service's `_KP_MIN`, and `CROP_PAD` is the padding in `_crop_worker`. Keep those three equal to the service, or the scores here stop describing live behaviour.

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

HAS_GPU = torch.cuda.is_available()
if HAS_GPU:
    YOLO_DEVICE = 0
else:
    YOLO_DEVICE = "cpu"

# Data
COCO_LABELS_URL = "https://github.com/ultralytics/assets/releases/download/v0.0.0/coco2017labels-pose.zip"
COCO_IMAGES_URL = "http://images.cocodataset.org/zips/val2017.zip"
COCO_SPLIT = "val2017"
MAX_SOURCE_IMAGES = None
VAL_FRACTION = 0.15
TEST_FRACTION = 0.15
MIN_PERSON_HEIGHT_PX = 64
MIN_LABELLED_KPTS = 5
OTHER_PERSON_MIN_INSIDE = 0.5
SITE_POSE_DIR = DRIVE_ROOT / "datasets" / "vision" / "pose" / "site"

# Must match the live service
CROP_PAD = 0.12
KP_MIN_CONF = 0.3
if PPE_METADATA.is_file():
    LIVE_CONF = float(json.loads(PPE_METADATA.read_text())["conf"])
    LIVE_CONF_SOURCE = str(PPE_METADATA)
else:
    LIVE_CONF = 0.25
    LIVE_CONF_SOURCE = "service fallback when the PPE metadata is missing"

# Training
MAX_EPOCHS = 30
PATIENCE = 5
STAGE_A_EPOCHS = 10
FREEZE_LAYERS = 10
if HAS_GPU:
    BATCH = 32
else:
    BATCH = 8
IMGSZ = 640
CAMERA_COPIES_PER_CROP = 1
SELECTION_METRIC = "service_pck"
WINNER_TOLERANCE = 0.005

# Evaluation
PCK_ALPHA = 0.10
BOOTSTRAP_ROUNDS = 1000
REPEAT_SEEDS = [1, 2]
TIMING_IMAGES = 100

config_rows = [
    {"setting": "SEED", "value": SEED, "meaning": "Random seed for splitting, augmentation, and training"},
    {"setting": "YOLO_DEVICE", "value": YOLO_DEVICE, "meaning": "0 is the first GPU, cpu means no GPU"},
    {"setting": "COCO_LABELS_URL", "value": COCO_LABELS_URL, "meaning": "Ultralytics pose labels for COCO 2017 (17 keypoints per person)"},
    {"setting": "COCO_IMAGES_URL", "value": COCO_IMAGES_URL, "meaning": "COCO 2017 val images. The stock model never trained on them"},
    {"setting": "MAX_SOURCE_IMAGES", "value": str(MAX_SOURCE_IMAGES), "meaning": "Cap on source photos for a quick run. None uses all"},
    {"setting": "VAL_FRACTION", "value": VAL_FRACTION, "meaning": "Share of source photos held out for val"},
    {"setting": "TEST_FRACTION", "value": TEST_FRACTION, "meaning": "Share of source photos held out for test"},
    {"setting": "MIN_PERSON_HEIGHT_PX", "value": MIN_PERSON_HEIGHT_PX, "meaning": "Smallest person box height, in source pixels, that becomes a crop"},
    {"setting": "MIN_LABELLED_KPTS", "value": MIN_LABELLED_KPTS, "meaning": "Fewest labelled joints a person needs to become a crop"},
    {"setting": "OTHER_PERSON_MIN_INSIDE", "value": OTHER_PERSON_MIN_INSIDE, "meaning": "A neighbour is labelled in a crop when at least this share of its box is inside"},
    {"setting": "SITE_POSE_DIR", "value": str(SITE_POSE_DIR), "meaning": "Optional own labelled SAMS crops (images/ and labels/). Added to train only"},
    {"setting": "CROP_PAD", "value": CROP_PAD, "meaning": "Padding around a worker box, same as _crop_worker"},
    {"setting": "KP_MIN_CONF", "value": KP_MIN_CONF, "meaning": "Joints below this confidence are missing, same as _KP_MIN"},
    {"setting": "LIVE_CONF", "value": LIVE_CONF, "meaning": f"Person confidence the service passes to pose. Source: {LIVE_CONF_SOURCE}"},
    {"setting": "MAX_EPOCHS", "value": MAX_EPOCHS, "meaning": "Upper limit on epochs. Early stopping usually ends sooner"},
    {"setting": "PATIENCE", "value": PATIENCE, "meaning": "Epochs without a val improvement before training stops"},
    {"setting": "STAGE_A_EPOCHS", "value": STAGE_A_EPOCHS, "meaning": "Epochs with the backbone frozen in two-stage fine-tuning"},
    {"setting": "FREEZE_LAYERS", "value": FREEZE_LAYERS, "meaning": "Layers frozen in stage A. 10 is the whole YOLOv8 backbone"},
    {"setting": "BATCH", "value": BATCH, "meaning": "Images per training step"},
    {"setting": "IMGSZ", "value": IMGSZ, "meaning": "Training and live image size. The service predicts at this size"},
    {"setting": "CAMERA_COPIES_PER_CROP", "value": CAMERA_COPIES_PER_CROP, "meaning": "Camera-damage copies made per training crop for E4"},
    {"setting": "SELECTION_METRIC", "value": SELECTION_METRIC, "meaning": "Val score used to choose: joints placed correctly the way the service reads them"},
    {"setting": "WINNER_TOLERANCE", "value": WINNER_TOLERANCE, "meaning": "Runs this close to the best score count as tied. Ties go to higher pose mAP50-95"},
    {"setting": "PCK_ALPHA", "value": PCK_ALPHA, "meaning": "A joint is correct within this share of the person box's longer side"},
    {"setting": "BOOTSTRAP_ROUNDS", "value": BOOTSTRAP_ROUNDS, "meaning": "Resampling rounds for the confidence intervals"},
    {"setting": "REPEAT_SEEDS", "value": str(REPEAT_SEEDS), "meaning": "Extra seeds used to retrain the chosen recipe"},
    {"setting": "TIMING_IMAGES", "value": TIMING_IMAGES, "meaning": "Single-image predictions timed in the export check"},
]
config_table = pd.DataFrame(config_rows).set_index("setting")
config_table.to_csv(RESULTS_DIR / "config.csv")
display(config_table)

### 0.4 Keypoint layout

The 17 COCO keypoints in COCO order. `FLIP_IDX` tells Ultralytics which joint swaps with which when an image is mirrored during training (left wrist becomes right wrist). The cell stops if an index the service relies on does not hold the joint the service expects, because the service reads joints by position, not by name.

In [ ]:
KEYPOINT_NAMES = [
    "nose", "left_eye", "right_eye", "left_ear", "right_ear",
    "left_shoulder", "right_shoulder", "left_elbow", "right_elbow",
    "left_wrist", "right_wrist", "left_hip", "right_hip",
    "left_knee", "right_knee", "left_ankle", "right_ankle",
]
NUM_KEYPOINTS = len(KEYPOINT_NAMES)
FLIP_IDX = [0, 2, 1, 4, 3, 6, 5, 8, 7, 10, 9, 12, 11, 14, 13, 16, 15]
SKELETON = [
    (15, 13), (13, 11), (16, 14), (14, 12), (11, 12), (5, 11), (6, 12), (5, 6),
    (5, 7), (6, 8), (7, 9), (8, 10), (1, 2), (0, 1), (0, 2), (1, 3), (2, 4), (3, 5), (4, 6),
]
JOINT_GROUPS = {
    "head": [0, 1, 2, 3, 4],
    "shoulders": [5, 6],
    "elbows": [7, 8],
    "wrists": [9, 10],
    "hips": [11, 12],
    "knees": [13, 14],
    "ankles": [15, 16],
}
GROUP_COLORS = {
    "head": "#d53f8c",
    "shoulders": "#dd6b20",
    "elbows": "#d69e2e",
    "wrists": "#38a169",
    "hips": "#3182ce",
    "knees": "#805ad5",
    "ankles": "#2c7a7b",
}
JOINT_GROUP_OF = {}
for group_name, indices in JOINT_GROUPS.items():
    for index in indices:
        JOINT_GROUP_OF[index] = group_name

HEAD_IDX = (0, 1, 2, 3, 4)
NOSE, L_EYE, R_EYE = 0, 1, 2
L_SHO, R_SHO = 5, 6
L_ANK, R_ANK = 15, 16

SERVICE_USE = {
    0: "head region, stature (nose)",
    1: "head region, stature (eyes)",
    2: "head region, stature (eyes)",
    3: "head region",
    4: "head region",
    5: "torso region, head-or-shoulders rule",
    6: "torso region, head-or-shoulders rule",
    7: "hand region direction",
    8: "hand region direction",
    9: "hand region",
    10: "hand region",
    11: "torso region",
    12: "torso region",
    15: "feet region, stature",
    16: "feet region, stature",
}
SERVICE_EXPECTED_NAMES = {
    0: "nose", 1: "left_eye", 2: "right_eye", 5: "left_shoulder", 6: "right_shoulder",
    7: "left_elbow", 8: "right_elbow", 9: "left_wrist", 10: "right_wrist",
    11: "left_hip", 12: "right_hip", 15: "left_ankle", 16: "right_ankle",
}
for index, expected in SERVICE_EXPECTED_NAMES.items():
    if KEYPOINT_NAMES[index] != expected:
        raise RuntimeError(f"Joint {index} is {KEYPOINT_NAMES[index]}, the service expects {expected}")
for index, partner in enumerate(FLIP_IDX):
    if FLIP_IDX[partner] != index:
        raise RuntimeError(f"FLIP_IDX is not symmetric at joint {index}")

layout_rows = []
for index, name in enumerate(KEYPOINT_NAMES):
    layout_rows.append({
        "index": index,
        "joint": name,
        "group": JOINT_GROUP_OF[index],
        "mirror partner": KEYPOINT_NAMES[FLIP_IDX[index]],
        "used by the service for": SERVICE_USE.get(index, "not used directly"),
    })
layout_table = pd.DataFrame(layout_rows).set_index("index")
layout_table.to_csv(RESULTS_DIR / "keypoint_layout.csv")
display(layout_table)

### 0.5 Helper functions

Every crop is written into one staging folder, `merged/`:

| Folder | Holds |
|--------|-------|
| `images/all` | One padded crop per person. Train, val, and test are chosen from here |
| `images/site` | Your own labelled SAMS crops, if `SITE_POSE_DIR` has any. Train only |
| `images/camera` | Camera-damage copies of training crops, made in section 6 |
| `images/robust_*` | Damaged copies of val crops for the robustness test in section 10 |
| `labels/...` | One YOLO pose label file per image, in the matching folder |

A YOLO pose label line is `class xc yc w h` followed by 17 joints of `x y v`, all normalised to the image, so a full line has 56 values. `v` is 0 for a joint that is not labelled, 1 for labelled but hidden, and 2 for labelled and visible. The first line of every crop's label file is the person the crop was cut for.

In [ ]:
IMAGE_SUFFIXES = {".jpg", ".jpeg", ".png", ".bmp"}
MERGED = DATA_DIR / "merged"
LISTS_DIR = MERGED / "lists"
VALUES_PER_LINE = 5 + NUM_KEYPOINTS * 3


def list_images(folder):
    found = []
    folder = Path(folder)
    if not folder.exists():
        return found
    for path in sorted(folder.iterdir()):
        if path.suffix.lower() in IMAGE_SUFFIXES:
            found.append(path)
    return found


def label_path_for(image_path):
    image_path = Path(image_path)
    folder = image_path.parent.name
    return MERGED / "labels" / folder / f"{image_path.stem}.txt"


def read_pose_labels(label_path):
    """Read a YOLO pose label file into a list of {"box": [xc, yc, w, h], "kpts": (17, 3) array}."""
    persons = []
    label_path = Path(label_path)
    if not label_path.is_file():
        return persons
    for line in label_path.read_text().splitlines():
        parts = line.split()
        if len(parts) < 5:
            continue
        values = [float(value) for value in parts]
        kpts = np.zeros((NUM_KEYPOINTS, 3), dtype=np.float64)
        if len(values) >= VALUES_PER_LINE:
            kpts = np.asarray(values[5:VALUES_PER_LINE], dtype=np.float64).reshape(NUM_KEYPOINTS, 3)
        persons.append({"cls": int(values[0]), "box": np.asarray(values[1:5]), "kpts": kpts})
    return persons


def write_pose_labels(label_path, persons):
    label_path = Path(label_path)
    label_path.parent.mkdir(parents=True, exist_ok=True)
    lines = []
    for person in persons:
        xc, yc, w, h = [float(value) for value in person["box"]]
        parts = [f"0 {xc:.6f} {yc:.6f} {w:.6f} {h:.6f}"]
        for x, y, v in person["kpts"]:
            parts.append(f"{x:.6f} {y:.6f} {int(v)}")
        lines.append(" ".join(parts))
    text = ""
    if lines:
        text = "\n".join(lines) + "\n"
    label_path.write_text(text)


def write_list(name, image_paths):
    """Write a text file with one image path per line, for Ultralytics."""
    LISTS_DIR.mkdir(parents=True, exist_ok=True)
    path = LISTS_DIR / f"{name}.txt"
    lines = []
    for image_path in image_paths:
        lines.append(str(image_path))
    path.write_text("\n".join(lines) + "\n")
    return path


def read_list(list_path):
    paths = []
    for line in Path(list_path).read_text().splitlines():
        if line.strip():
            paths.append(Path(line.strip()))
    return paths


def free_memory():
    gc.collect()
    if HAS_GPU:
        torch.cuda.empty_cache()


def save_figure(fig, name):
    path = RESULTS_DIR / f"{name}.png"
    fig.savefig(path, dpi=140, bbox_inches="tight")
    plt.show()
    print("Saved figure:", path)


def save_table(frame, name):
    frame.to_csv(RESULTS_DIR / f"{name}.csv")
    display(frame)


def draw_skeleton(ax, points, color=None, min_value=0.5, size=18):
    """Draw joints given as (17, 3) pixel rows of x, y, and visibility or confidence."""
    for a, b in SKELETON:
        if points[a, 2] >= min_value and points[b, 2] >= min_value:
            line_color = color or GROUP_COLORS[JOINT_GROUP_OF[a]]
            ax.plot([points[a, 0], points[b, 0]], [points[a, 1], points[b, 1]], color=line_color, linewidth=2)
    for index in range(NUM_KEYPOINTS):
        if points[index, 2] >= min_value:
            point_color = color or GROUP_COLORS[JOINT_GROUP_OF[index]]
            ax.scatter(points[index, 0], points[index, 1], s=size, color=point_color, edgecolors="white", linewidths=0.5, zorder=3)


def labels_to_pixels(person, width, height):
    points = person["kpts"].copy()
    points[:, 0] = points[:, 0] * width
    points[:, 1] = points[:, 1] * height
    return points


def draw_pose(ax, image_path, persons, title="", box_color="#2f855a"):
    """Draw boxes and labelled joints. The first person is the crop's own person (solid box)."""
    image = Image.open(image_path).convert("RGB")
    width, height = image.size
    ax.imshow(image)
    for number, person in enumerate(persons):
        xc, yc, w, h = person["box"]
        rectangle = patches.Rectangle(
            ((xc - w / 2) * width, (yc - h / 2) * height), w * width, h * height,
            fill=False, linewidth=2, edgecolor=box_color, linestyle="-" if number == 0 else ":",
        )
        ax.add_patch(rectangle)
        draw_skeleton(ax, labels_to_pixels(person, width, height))
    if title:
        ax.set_title(title, fontsize=9)
    ax.axis("off")


def preview_crops(image_paths, title, name, columns=4, rows=2):
    picks = [Path(path) for path in image_paths]
    if not picks:
        display(pd.DataFrame([{"preview": title, "result": "no images"}]))
        return
    random.Random(SEED).shuffle(picks)
    picks = picks[:columns * rows]
    fig, axes = plt.subplots(rows, columns, figsize=(3.6 * columns, 4.4 * rows), squeeze=False)
    for ax in axes.flatten():
        ax.axis("off")
    for ax, image_path in zip(axes.flatten(), picks):
        persons = read_pose_labels(label_path_for(image_path))
        labelled = int((persons[0]["kpts"][:, 2] > 0).sum()) if persons else 0
        draw_pose(ax, image_path, persons, title=f"{image_path.stem[:22]}  ({labelled} joints)")
    fig.suptitle(title)
    plt.tight_layout()
    save_figure(fig, name)


def brightness_contrast(image):
    grey = np.asarray(image.convert("L").resize((64, 64)), dtype=np.float32)
    return float(grey.mean()), float(grey.std())


if MERGED.exists():
    shutil.rmtree(MERGED)
MERGED.mkdir(parents=True)
print("Fresh staging folder:", MERGED)

## 1. Download COCO val 2017 and the pose labels

Two files are downloaded once and reused on later runs:

- the Ultralytics conversion of the COCO keypoint labels (`coco2017labels-pose.zip`), of which only the `val2017` labels are unpacked,
- the 5,000 COCO val 2017 images (`val2017.zip`, about 800 MB).

Why val 2017 only: the stock `yolov8n-pose.pt` was trained on COCO train 2017. Val 2017 photos are new to it, so the stock model can be scored fairly on the same held-out crops as the fine-tuned models. The full train 2017 set is about 19 GB and is left out to keep the notebook runnable in Colab.

In [ ]:
RAW_DIR = DATA_DIR / "coco_raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)


def fetch(url, target):
    target = Path(target)
    if target.is_file() and target.stat().st_size > 0:
        print("Already downloaded:", target.name)
        return target
    partial = target.with_suffix(target.suffix + ".part")
    print("Downloading", url)
    urllib.request.urlretrieve(url, partial)
    partial.rename(target)
    return target


labels_zip = fetch(COCO_LABELS_URL, RAW_DIR / "coco2017labels-pose.zip")
images_zip = fetch(COCO_IMAGES_URL, RAW_DIR / f"{COCO_SPLIT}.zip")

LABELS_UNZIPPED = RAW_DIR / "labels_unzipped"
IMAGES_UNZIPPED = RAW_DIR / "images_unzipped"
if not (LABELS_UNZIPPED / ".done").exists():
    with zipfile.ZipFile(labels_zip) as archive:
        members = [name for name in archive.namelist() if f"labels/{COCO_SPLIT}/" in name]
        archive.extractall(LABELS_UNZIPPED, members=members)
    (LABELS_UNZIPPED / ".done").write_text("ok")
if not (IMAGES_UNZIPPED / ".done").exists():
    with zipfile.ZipFile(images_zip) as archive:
        archive.extractall(IMAGES_UNZIPPED)
    (IMAGES_UNZIPPED / ".done").write_text("ok")

label_dirs = [path for path in LABELS_UNZIPPED.rglob(COCO_SPLIT) if path.is_dir() and path.parent.name == "labels"]
image_dirs = [path for path in IMAGES_UNZIPPED.rglob(COCO_SPLIT) if path.is_dir()]
if not label_dirs:
    raise FileNotFoundError(f"No labels/{COCO_SPLIT} folder inside {labels_zip}")
if not image_dirs:
    raise FileNotFoundError(f"No {COCO_SPLIT} image folder inside {images_zip}")
SOURCE_LABELS = label_dirs[0]
SOURCE_IMAGES = image_dirs[0]

download_table = pd.DataFrame([
    {"item": "label folder", "path": str(SOURCE_LABELS), "files": len(list(SOURCE_LABELS.glob("*.txt")))},
    {"item": "image folder", "path": str(SOURCE_IMAGES), "files": len(list_images(SOURCE_IMAGES))},
]).set_index("item")
save_table(download_table, "download_files")

The label check below reads every label file once. Every line must have 56 values (box plus 17 joints of three values). If the format ever changes, the notebook stops here instead of training on misread joints. Only photos with at least one labelled person have a label file, so photos without people are not used.

In [ ]:
source_images = list_images(SOURCE_IMAGES)
values_per_line = Counter()
visibility_counts = Counter()
persons_per_photo = Counter()
SOURCE_PERSONS = {}

for image_path in source_images:
    label_file = SOURCE_LABELS / f"{image_path.stem}.txt"
    if not label_file.is_file():
        continue
    for line in label_file.read_text().splitlines():
        if line.strip():
            values_per_line[len(line.split())] += 1
    persons = read_pose_labels(label_file)
    if not persons:
        continue
    SOURCE_PERSONS[image_path] = persons
    persons_per_photo[len(persons)] += 1
    for person in persons:
        for v in person["kpts"][:, 2]:
            visibility_counts[int(v)] += 1

bad_lengths = {length: count for length, count in values_per_line.items() if length != VALUES_PER_LINE}
if bad_lengths:
    raise RuntimeError(f"Label lines without {VALUES_PER_LINE} values: {bad_lengths}")

summary_rows = [
    {"item": "source photos", "value": len(source_images)},
    {"item": "photos with labelled people", "value": len(SOURCE_PERSONS)},
    {"item": "labelled people", "value": int(sum(len(p) for p in SOURCE_PERSONS.values()))},
    {"item": "values per label line", "value": str(dict(values_per_line))},
    {"item": "joints not labelled (v=0)", "value": visibility_counts[0]},
    {"item": "joints labelled, hidden (v=1)", "value": visibility_counts[1]},
    {"item": "joints labelled, visible (v=2)", "value": visibility_counts[2]},
]
save_table(pd.DataFrame(summary_rows).set_index("item"), "source_label_summary")

## 2. Cut one crop per person, the way the service does

For every labelled person, the box is padded by `CROP_PAD` on each side and cropped, with the same rounding and image-border clipping as `_crop_worker`. The crop becomes a new training image. Its labels are moved into crop coordinates:

- the crop's own person is always the first label line,
- a neighbour is also labelled if at least `OTHER_PERSON_MIN_INSIDE` of its box is inside the crop, so the model is not taught to ignore a person who is really there,
- a joint that falls outside the crop is set to `0 0 0` (not labelled).

A person becomes a crop only if its box is at least `MIN_PERSON_HEIGHT_PX` tall and it has at least `MIN_LABELLED_KPTS` labelled joints. Tiny or barely labelled people would not get past the PPE detector in SAMS, and they carry little joint information. The filter table shows how many people each rule removed.

In [ ]:
def crop_window(box_px, width, height, pad_frac=CROP_PAD):
    """Same arithmetic as vision_inference._crop_worker."""
    x1, y1, x2, y2 = box_px
    pad_x = max(1.0, (x2 - x1) * pad_frac)
    pad_y = max(1.0, (y2 - y1) * pad_frac)
    left = max(0, int(x1 - pad_x))
    top = max(0, int(y1 - pad_y))
    right = min(width, int(x2 + pad_x))
    bottom = min(height, int(y2 + pad_y))
    if right - left < 8 or bottom - top < 8:
        return None
    return left, top, right, bottom


def box_to_pixels(box, width, height):
    xc, yc, w, h = box
    return [(xc - w / 2) * width, (yc - h / 2) * height, (xc + w / 2) * width, (yc + h / 2) * height]


def person_in_crop(person, width, height, window):
    """Move one person's box and joints into crop coordinates. None if too little is inside."""
    left, top, right, bottom = window
    crop_w = right - left
    crop_h = bottom - top
    x1, y1, x2, y2 = box_to_pixels(person["box"], width, height)
    area = max(1e-6, (x2 - x1) * (y2 - y1))
    cx1, cy1 = max(x1, left), max(y1, top)
    cx2, cy2 = min(x2, right), min(y2, bottom)
    if cx2 <= cx1 or cy2 <= cy1:
        return None, 0.0
    inside = (cx2 - cx1) * (cy2 - cy1) / area
    box = [
        ((cx1 + cx2) / 2 - left) / crop_w,
        ((cy1 + cy2) / 2 - top) / crop_h,
        (cx2 - cx1) / crop_w,
        (cy2 - cy1) / crop_h,
    ]
    kpts = np.zeros((NUM_KEYPOINTS, 3))
    for index in range(NUM_KEYPOINTS):
        kx, ky, v = person["kpts"][index]
        if v <= 0:
            continue
        px = kx * width - left
        py = ky * height - top
        if 0 <= px < crop_w and 0 <= py < crop_h:
            kpts[index] = [px / crop_w, py / crop_h, v]
    return {"box": np.asarray(box), "kpts": kpts}, inside


photos = sorted(SOURCE_PERSONS)
if MAX_SOURCE_IMAGES is not None:
    random.Random(SEED).shuffle(photos)
    photos = sorted(photos[:MAX_SOURCE_IMAGES])

filter_counts = Counter()
crop_rows = []
crop_dir = MERGED / "images" / "all"
crop_dir.mkdir(parents=True, exist_ok=True)

for photo in photos:
    persons = SOURCE_PERSONS[photo]
    with Image.open(photo) as opened:
        image = opened.convert("RGB")
    width, height = image.size
    for index, person in enumerate(persons):
        filter_counts["people"] += 1
        box_px = box_to_pixels(person["box"], width, height)
        person_h = box_px[3] - box_px[1]
        labelled = int((person["kpts"][:, 2] > 0).sum())
        if person_h < MIN_PERSON_HEIGHT_PX:
            filter_counts["too small"] += 1
            continue
        if labelled < MIN_LABELLED_KPTS:
            filter_counts["too few labelled joints"] += 1
            continue
        window = crop_window(box_px, width, height)
        if window is None:
            filter_counts["crop under 8 pixels"] += 1
            continue
        main, _ = person_in_crop(person, width, height, window)
        crop_persons = [main]
        neighbours = 0
        for other_index, other in enumerate(persons):
            if other_index == index:
                continue
            moved, inside = person_in_crop(other, width, height, window)
            if moved is not None and inside >= OTHER_PERSON_MIN_INSIDE:
                crop_persons.append(moved)
                neighbours += 1
        crop = image.crop(window)
        crop_path = crop_dir / f"{photo.stem}_p{index:02d}.jpg"
        crop.save(crop_path, quality=95)
        write_pose_labels(label_path_for(crop_path), crop_persons)
        brightness, contrast = brightness_contrast(crop)
        filter_counts["kept"] += 1
        crop_rows.append({
            "image": str(crop_path),
            "source": photo.stem,
            "person_index": index,
            "crop_w": crop.width,
            "crop_h": crop.height,
            "person_h_px": person_h,
            "labelled": labelled,
            "visible": int((main["kpts"][:, 2] == 2).sum()),
            "labelled_in_crop": int((main["kpts"][:, 2] > 0).sum()),
            "neighbours": neighbours,
            "brightness": brightness,
            "contrast": contrast,
        })

manifest = pd.DataFrame(crop_rows)
manifest["upscale_to_imgsz"] = IMGSZ / manifest[["crop_w", "crop_h"]].max(axis=1)
manifest["aspect"] = manifest["crop_w"] / manifest["crop_h"]

filter_table = pd.DataFrame([{"rule": rule, "people": count} for rule, count in filter_counts.items()]).set_index("rule")
save_table(filter_table, "crop_filter_counts")
print(f"{len(manifest)} crops from {manifest['source'].nunique()} photos")

One source photo with the crop windows drawn on it, then the crops cut from it. The dashed box on a crop marks a labelled neighbour.

In [ ]:
busy_source = manifest["source"].value_counts().index[0]
busy_photo = next(photo for photo in photos if photo.stem == busy_source)
fig, ax = plt.subplots(figsize=(9, 7))
with Image.open(busy_photo) as opened:
    full = opened.convert("RGB")
ax.imshow(full)
for _, row in manifest[manifest["source"] == busy_source].iterrows():
    person = SOURCE_PERSONS[busy_photo][row["person_index"]]
    window = crop_window(box_to_pixels(person["box"], full.width, full.height), full.width, full.height)
    left, top, right, bottom = window
    ax.add_patch(patches.Rectangle((left, top), right - left, bottom - top, fill=False, edgecolor="#e53e3e", linewidth=2))
    draw_skeleton(ax, labels_to_pixels(person, full.width, full.height))
ax.set_title(f"{busy_source}: crop windows (red) with {CROP_PAD:.0%} padding")
ax.axis("off")
save_figure(fig, "crop_windows_example")

busy_crops = manifest[manifest["source"] == busy_source]["image"].tolist()
preview_crops(busy_crops, f"Crops cut from {busy_source}", "crop_examples", columns=min(4, len(busy_crops)), rows=1)

## 3. Train, val, and test splits by source photo

Crops from one photo share the same scene, lighting, and often the same people. If one crop of a photo were in train and another in test, the test would be easier than real life. So whole source photos are split **70 / 15 / 15** with seed 42, and every crop follows its photo. The cell stops if any photo ends up in two splits.

Your own labelled SAMS crops in `SITE_POSE_DIR` (if any) go into train only. Scoring stays on COCO crops, so val and test never contain them.

In [ ]:
sources = np.array(sorted(manifest["source"].unique()))
order = np.random.default_rng(SEED).permutation(len(sources))
test_count = int(round(len(sources) * TEST_FRACTION))
val_count = int(round(len(sources) * VAL_FRACTION))
test_sources = set(sources[order[:test_count]])
val_sources = set(sources[order[test_count:test_count + val_count]])

manifest["split"] = "train"
manifest.loc[manifest["source"].isin(val_sources), "split"] = "val"
manifest.loc[manifest["source"].isin(test_sources), "split"] = "test"

split_sources = manifest.groupby("split")["source"].apply(set)
for first in ["train", "val", "test"]:
    for second in ["train", "val", "test"]:
        if first < second and split_sources.get(first, set()) & split_sources.get(second, set()):
            raise RuntimeError(f"Photos shared between {first} and {second}")

TRAIN_IMAGES = [Path(path) for path in manifest.loc[manifest["split"] == "train", "image"]]
VAL_IMAGES = [Path(path) for path in manifest.loc[manifest["split"] == "val", "image"]]
TEST_IMAGES = [Path(path) for path in manifest.loc[manifest["split"] == "test", "image"]]

SITE_IMAGES = []
site_images_dir = SITE_POSE_DIR / "images"
site_labels_dir = SITE_POSE_DIR / "labels"
for image_path in list_images(site_images_dir):
    source_label = site_labels_dir / f"{image_path.stem}.txt"
    if not source_label.is_file():
        continue
    lengths = {len(line.split()) for line in source_label.read_text().splitlines() if line.strip()}
    if lengths and lengths != {VALUES_PER_LINE}:
        raise RuntimeError(f"{source_label} has lines without {VALUES_PER_LINE} values")
    target = MERGED / "images" / "site" / image_path.name
    target.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(image_path, target)
    label_target = label_path_for(target)
    label_target.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(source_label, label_target)
    SITE_IMAGES.append(target)

VAL_LIST = write_list("val", VAL_IMAGES)
TEST_LIST = write_list("test", TEST_IMAGES)

split_rows = []
for split_name in ["train", "val", "test"]:
    subset = manifest[manifest["split"] == split_name]
    split_rows.append({
        "split": split_name,
        "source photos": subset["source"].nunique(),
        "crops": len(subset),
        "labelled joints": int(subset["labelled_in_crop"].sum()),
        "share of crops": round(len(subset) / len(manifest), 3),
    })
split_rows.append({"split": "site (train only)", "source photos": len(SITE_IMAGES), "crops": len(SITE_IMAGES),
                   "labelled joints": None, "share of crops": None})
save_table(pd.DataFrame(split_rows).set_index("split"), "splits")

### Preview of the crops

Eight random crops from each split with their labels. Check that the joints sit on the right body parts, that the left and right sides are not swapped, and that a neighbour who is clearly in the crop has a dashed box.

In [ ]:
preview_crops(TRAIN_IMAGES, "Train crops", "preview_train")
preview_crops(VAL_IMAGES, "Val crops", "preview_val")
preview_crops(TEST_IMAGES, "Test crops", "preview_test")

## 4. Exploratory data analysis

The EDA answers five questions before any training:

1. How much data is in each split, and how crowded are the crops?
2. Which joints are labelled, and which are usually missing? A joint that is rarely labelled is also rarely learned.
3. How big are the crops, and how much does the service have to enlarge them to reach `IMGSZ`?
4. How bright and how contrasted are the crops? SAMS cameras are often worse than COCO photos.
5. Where do the joints sit inside a crop?

The EDA uses every crop in all splits, unless a cell says otherwise.

### 4.1 Crops per split and people per crop

The left chart counts crops. The right chart counts how many labelled people each crop holds (its own person plus neighbours). A crop with several people is the hard case for the service, because `_joints_inside` keeps the highest-scoring pose that passes the head-or-shoulders rule, which may belong to a neighbour.

In [ ]:
manifest["people in crop"] = 1 + manifest["neighbours"]
crops_per_split = manifest["split"].value_counts().reindex(["train", "val", "test"], fill_value=0)
people_table = pd.crosstab(manifest["people in crop"].clip(upper=4), manifest["split"])
people_table = people_table.reindex(columns=["train", "val", "test"], fill_value=0)
people_table.index = [str(value) if value < 4 else "4 or more" for value in people_table.index]
people_table.index.name = "people in crop"
save_table(people_table, "eda_people_per_crop")

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
crops_per_split.plot(kind="bar", ax=axes[0], color=["#2f855a", "#2b6cb0", "#6b46c1"])
axes[0].set_title("Crops per split")
axes[0].set_ylabel("crops")
axes[0].tick_params(axis="x", rotation=0)
(people_table / people_table.sum() * 100).plot(kind="bar", ax=axes[1])
axes[1].set_title("People per crop, percent of each split")
axes[1].set_ylabel("percent of crops")
axes[1].tick_params(axis="x", rotation=0)
plt.tight_layout()
save_figure(fig, "eda_crops_and_people")

### 4.2 Which joints are labelled

For the crop's own person, each bar splits the crops into joints labelled and visible (`v=2`), labelled but hidden (`v=1`), and not labelled (`v=0`, out of view or not annotated). The second table counts the crops on which the labels themselves would satisfy two service rules: the head-or-shoulders rule, and the head-to-ankles stature that distance estimation needs. If stature is possible on few crops, the distance network will often run without it in SAMS too.

In [ ]:
joint_rows = []
for _, row in manifest.iterrows():
    main = read_pose_labels(label_path_for(row["image"]))[0]
    for index in range(NUM_KEYPOINTS):
        x, y, v = main["kpts"][index]
        joint_rows.append({
            "image": row["image"],
            "split": row["split"],
            "joint": KEYPOINT_NAMES[index],
            "index": index,
            "group": JOINT_GROUP_OF[index],
            "v": int(v),
            "x": x,
            "y": y,
        })
joints = pd.DataFrame(joint_rows)

visibility = pd.crosstab(joints["joint"], joints["v"], normalize="index") * 100
visibility = visibility.reindex(index=KEYPOINT_NAMES, columns=[2, 1, 0], fill_value=0)
visibility.columns = ["visible (v=2)", "hidden (v=1)", "not labelled (v=0)"]
save_table(visibility.round(1), "eda_joint_labelling_percent")

fig, ax = plt.subplots(figsize=(10, 6))
visibility.iloc[::-1].plot(kind="barh", stacked=True, ax=ax, color=["#2f855a", "#d69e2e", "#a0aec0"])
ax.set_xlabel("percent of crops")
ax.set_title("Labelling of each joint on the crop's own person")
ax.legend(loc="lower right")
plt.tight_layout()
save_figure(fig, "eda_joint_labelling")

labelled_matrix = np.zeros((len(manifest), NUM_KEYPOINTS), dtype=bool)
for position, (_, row) in enumerate(manifest.iterrows()):
    main = read_pose_labels(label_path_for(row["image"]))[0]
    labelled_matrix[position] = main["kpts"][:, 2] > 0
head_or_shoulders = labelled_matrix[:, list(HEAD_IDX)].any(axis=1) | labelled_matrix[:, L_SHO] | labelled_matrix[:, R_SHO]
head_for_stature = labelled_matrix[:, NOSE] | (labelled_matrix[:, L_EYE] & labelled_matrix[:, R_EYE])
stature_possible = head_for_stature & labelled_matrix[:, L_ANK] & labelled_matrix[:, R_ANK]
manifest["labels pass head-or-shoulders"] = head_or_shoulders
manifest["labels allow stature"] = stature_possible

rule_table = manifest.groupby("split")[["labels pass head-or-shoulders", "labels allow stature"]].mean() * 100
rule_table = rule_table.reindex(["train", "val", "test"]).round(1)
save_table(rule_table, "eda_service_rules_on_labels_percent")

### 4.3 Crop size

The left chart is the person height in source pixels, on a log scale. The right chart is how much the service enlarges each crop to reach `IMGSZ` pixels on its longer side. A large enlargement means a blurry input, which matters because SAMS workers far from the camera produce small crops too.

In [ ]:
size_table = manifest.groupby("split")[["crop_w", "crop_h", "person_h_px", "upscale_to_imgsz", "aspect"]].describe(
    percentiles=[0.1, 0.5, 0.9]
)
size_table = size_table.loc[:, (slice(None), ["10%", "50%", "90%"])].round(2)
save_table(size_table, "eda_crop_size")

fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))
sns.histplot(data=manifest, x="person_h_px", hue="split", hue_order=["train", "val", "test"],
             log_scale=True, element="step", stat="density", common_norm=False, ax=axes[0])
axes[0].set_title("Person height in source pixels (log)")
sns.histplot(data=manifest, x="upscale_to_imgsz", hue="split", hue_order=["train", "val", "test"],
             log_scale=True, element="step", stat="density", common_norm=False, ax=axes[1])
axes[1].axvline(1.0, color="grey", linestyle="--")
axes[1].set_title(f"Enlargement to {IMGSZ} pixels (log, right of 1 means enlarged)")
plt.tight_layout()
save_figure(fig, "eda_crop_size")

### 4.4 Brightness and contrast

Mean and standard deviation of the grey-level pixels of each crop. COCO photos are mostly well lit. If the spread is narrow, the robustness test in section 10 (dark, blurred, compressed, low-resolution crops) is the main evidence of how the model will behave on SAMS cameras.

In [ ]:
look_table = manifest.groupby("split")[["brightness", "contrast"]].describe(percentiles=[0.1, 0.5, 0.9])
look_table = look_table.loc[:, (slice(None), ["mean", "10%", "50%", "90%"])].round(1)
save_table(look_table, "eda_brightness_contrast")

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
sns.boxplot(data=manifest, x="split", y="brightness", order=["train", "val", "test"], ax=axes[0])
axes[0].set_title("Brightness per crop (0 black, 255 white)")
sns.boxplot(data=manifest, x="split", y="contrast", order=["train", "val", "test"], ax=axes[1])
axes[1].set_title("Contrast per crop")
plt.tight_layout()
save_figure(fig, "eda_brightness_contrast")

### 4.5 Where joints sit in a crop

Every labelled joint of the crop's own person, in crop coordinates (top of the crop at the top). Because the crop is cut around the person, joints should form a body-shaped cloud: head near the top, ankles near the bottom. Joints far outside that cloud are unusual poses (bending, lying) or label errors.

In [ ]:
labelled_joints = joints[joints["v"] > 0]
fig, ax = plt.subplots(figsize=(6.5, 8))
for group_name, color in GROUP_COLORS.items():
    subset = labelled_joints[labelled_joints["group"] == group_name]
    if len(subset) > 4000:
        subset = subset.sample(4000, random_state=SEED)
    ax.scatter(subset["x"], subset["y"], s=3, alpha=0.25, color=color, label=group_name)
ax.set_xlim(0, 1)
ax.set_ylim(1, 0)
ax.set_xlabel("x in crop")
ax.set_ylabel("y in crop")
ax.set_title("Labelled joints of the crop's own person")
ax.legend(markerscale=4, fontsize=8, loc="lower right")
plt.tight_layout()
save_figure(fig, "eda_joint_positions")

position_table = labelled_joints.groupby("group")[["x", "y"]].agg(["mean", "std"]).reindex(list(JOINT_GROUPS)).round(3)
save_table(position_table, "eda_joint_positions_by_group")

## 5. PCA of body shapes

PCA (principal component analysis) turns many correlated numbers into a few new axes, ordered by how much of the variation they explain. Here each crop's own person is described by 34 numbers: the x and y of its 17 joints, measured from the box center and divided by the box height, so a near and a far person with the same pose get the same numbers.

Only people with at least 13 labelled joints are used, so that the shape is mostly real. Missing joints are filled with the training mean of that joint. Scaling and PCA are fitted on train only, then val and test are projected onto the same axes. If val or test points sit where there are no training points, those poses are unlike anything in training.

In [ ]:
shape_rows = []
shape_info = []
for _, row in manifest.iterrows():
    main = read_pose_labels(label_path_for(row["image"]))[0]
    labelled = main["kpts"][:, 2] > 0
    if labelled.sum() < 13:
        continue
    xc, yc, w, h = main["box"]
    crop_aspect = row["crop_w"] / row["crop_h"]
    x = (main["kpts"][:, 0] - xc) * crop_aspect / max(h, 1e-6)
    y = (main["kpts"][:, 1] - yc) / max(h, 1e-6)
    x[~labelled] = np.nan
    y[~labelled] = np.nan
    shape_rows.append(np.concatenate([x, y]))
    shape_info.append({"image": row["image"], "split": row["split"], "labelled": int(labelled.sum()),
                       "box aspect": (w * row["crop_w"]) / max(h * row["crop_h"], 1e-6)})

shape_matrix = np.vstack(shape_rows)
shape_info = pd.DataFrame(shape_info)
is_train = (shape_info["split"] == "train").to_numpy()
fill_values = np.nanmean(shape_matrix[is_train], axis=0)
filled = np.where(np.isnan(shape_matrix), fill_values, shape_matrix)

scaler = StandardScaler().fit(filled[is_train])
shape_pca = PCA(n_components=5, random_state=SEED).fit(scaler.transform(filled[is_train]))
projected = shape_pca.transform(scaler.transform(filled))
shape_info["PC1"] = projected[:, 0]
shape_info["PC2"] = projected[:, 1]

variance_rows = []
cumulative = 0.0
for index, ratio in enumerate(shape_pca.explained_variance_ratio_):
    cumulative += ratio
    variance_rows.append({"component": f"PC{index + 1}", "explained %": ratio * 100, "cumulative %": cumulative * 100})
save_table(pd.DataFrame(variance_rows).set_index("component").round(1), "pca_shape_explained_variance")

feature_names = [f"{name} x" for name in KEYPOINT_NAMES] + [f"{name} y" for name in KEYPOINT_NAMES]
loadings = pd.DataFrame(shape_pca.components_[:2].T, index=feature_names, columns=["PC1", "PC2"])
top_loadings = []
for component in ["PC1", "PC2"]:
    strongest = loadings[component].abs().sort_values(ascending=False).head(6).index
    for feature in strongest:
        top_loadings.append({"component": component, "feature": feature, "loading": loadings.loc[feature, component]})
save_table(pd.DataFrame(top_loadings).set_index(["component", "feature"]).round(3), "pca_shape_top_loadings")

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
sns.scatterplot(data=shape_info, x="PC1", y="PC2", hue="split", hue_order=["train", "val", "test"],
                s=10, alpha=0.5, ax=axes[0])
axes[0].set_title(f"Body-shape PCA by split ({len(shape_info)} people)")
sns.scatterplot(data=shape_info, x="PC1", y="PC2", hue=np.log(shape_info["box aspect"]), palette="coolwarm",
                s=10, alpha=0.6, ax=axes[1], legend=False)
axes[1].set_title("Same points, colored by log box aspect (red is wide: bending or lying)")
plt.tight_layout()
save_figure(fig, "pca_body_shape")

How to read it: the loadings table says what each axis means. For example, PC1 driven by the ankle and knee `y` values means the main variation is how much of the legs is visible or how much the person bends. Val and test should cover the same cloud as train. A small separate cluster of wide boxes is people lying or crouching, which matters for SAMS because a fallen worker should still get joints.

## 6. Build the training sets

Two training lists are written:

| List | Contents | Used by |
|------|----------|---------|
| `train` | Every train crop, plus your site crops if any | E1 to E3, and E5 if camera copies lose |
| `train_camera` | `train` plus `CAMERA_COPIES_PER_CROP` damaged copies of each train crop | E4, and E5 if camera copies win |

A camera-damage copy applies one of: motion blur, darkening, heavy JPEG compression, downscaling, or sensor noise. None of these move pixels, so the joint labels are copied unchanged. The settings are milder than the robustness test in section 10, so that test still measures damage stronger than anything seen in training. Val and test are never damaged.

In [ ]:
TRAIN_PATHS = list(TRAIN_IMAGES) + list(SITE_IMAGES)
TRAIN_LIST = write_list("train", TRAIN_PATHS)

CAMERA_AUGMENT = A.Compose([
    A.OneOf([
        A.MotionBlur(blur_limit=(3, 7), p=1.0),
        A.RandomBrightnessContrast(brightness_limit=(-0.30, -0.15), contrast_limit=(-0.2, 0.0), p=1.0),
        A.ImageCompression(quality_range=(30, 60), p=1.0),
        A.Downscale(scale_range=(0.5, 0.75), p=1.0),
        A.GaussNoise(std_range=(0.02, 0.06), p=1.0),
    ], p=1.0),
], seed=SEED)

camera_dir = MERGED / "images" / "camera"
camera_dir.mkdir(parents=True, exist_ok=True)
CAMERA_PATHS = []
for image_path in TRAIN_PATHS:
    with Image.open(image_path) as opened:
        pixels = np.asarray(opened.convert("RGB"))
    for copy_number in range(CAMERA_COPIES_PER_CROP):
        damaged = CAMERA_AUGMENT(image=pixels)["image"]
        target = camera_dir / f"{image_path.stem}_cam{copy_number}.jpg"
        Image.fromarray(damaged).save(target, quality=95)
        label_target = label_path_for(target)
        label_target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(label_path_for(image_path), label_target)
        CAMERA_PATHS.append(target)

TRAIN_CAMERA_LIST = write_list("train_camera", TRAIN_PATHS + CAMERA_PATHS)
preview_crops(CAMERA_PATHS, "Camera-damage copies of train crops", "camera_copies")

### Dataset YAML files for Ultralytics

`kpt_shape: [17, 3]` tells Ultralytics each person has 17 joints of three values, and `flip_idx` is the mirror table from 0.4. Both training YAMLs point at the same val and test lists.

In [ ]:
def write_data_yaml(name, train_list):
    data = {
        "path": str(MERGED),
        "train": str(train_list),
        "val": str(VAL_LIST),
        "test": str(TEST_LIST),
        "kpt_shape": [NUM_KEYPOINTS, 3],
        "flip_idx": FLIP_IDX,
        "names": {0: "person"},
    }
    path = DATA_DIR / f"pose_{name}.yaml"
    path.write_text(yaml.safe_dump(data, sort_keys=False))
    return path


DATA_YAML = write_data_yaml("main", TRAIN_LIST)
DATA_YAML_CAMERA = write_data_yaml("camera", TRAIN_CAMERA_LIST)
yaml_summary = pd.DataFrame([
    {"yaml": DATA_YAML.name, "train images": len(TRAIN_PATHS), "val images": len(VAL_IMAGES), "test images": len(TEST_IMAGES)},
    {"yaml": DATA_YAML_CAMERA.name, "train images": len(TRAIN_PATHS) + len(CAMERA_PATHS), "val images": len(VAL_IMAGES), "test images": len(TEST_IMAGES)},
]).set_index("yaml")
save_table(yaml_summary, "dataset_yaml")

## 7. Training setup

### Transfer learning and fine-tuning

Every trained model starts from the stock YOLOv8 pose weights, which already know human joints from COCO train 2017. Fine-tuning only has to adapt them to tight single-worker crops. That is why the learning rates below are lower than the YOLOv8 defaults for training from scratch (0.002 instead of 0.01 for SGD): a large step would first destroy what the model already knows.

YOLOv8 has three parts: the **backbone** (layers 0 to 9) turns the image into features, the **neck** mixes features at different scales, and the **head** predicts boxes and joints. Freezing a layer means its weights are not updated.

| Method | What happens |
|--------|--------------|
| single stage | All layers train from the first epoch |
| two stage | Stage A freezes the backbone for `STAGE_A_EPOCHS` epochs, so only neck and head adapt. Stage B unfreezes everything at a tenth of the learning rate |

`mosaic` is 0: mosaic glues four images into one, which would show the model four small people instead of the one large worker the service sends. `pose` and `kobj` are the weights of the joint position loss and the joint visibility loss.

In [ ]:
BASE_HPARAMS = {
    "optimizer": "SGD",
    "lr0": 0.002,
    "lrf": 0.1,
    "momentum": 0.937,
    "weight_decay": 0.0005,
    "warmup_epochs": 1.0,
    "cos_lr": False,
    "box": 7.5,
    "cls": 0.5,
    "dfl": 1.5,
    "pose": 12.0,
    "kobj": 1.0,
    "hsv_h": 0.015,
    "hsv_s": 0.7,
    "hsv_v": 0.4,
    "degrees": 0.0,
    "translate": 0.1,
    "scale": 0.3,
    "fliplr": 0.5,
    "flipud": 0.0,
    "mosaic": 0.0,
    "close_mosaic": 0,
}

HPARAM_MEANING = {
    "optimizer": "Update rule. SGD with momentum is the YOLOv8 reference setting",
    "lr0": "Starting learning rate, low because the weights are already trained for pose",
    "lrf": "Final learning rate as a fraction of lr0",
    "momentum": "How much of the last update carries into the next one",
    "weight_decay": "Penalty on large weights, reduces overfitting",
    "warmup_epochs": "Epochs where the learning rate ramps up from a small value",
    "cos_lr": "Cosine learning-rate decay instead of linear",
    "box": "Weight of the box position loss",
    "cls": "Weight of the class loss (one class, person)",
    "dfl": "Weight of the distribution focal loss for box edges",
    "pose": "Weight of the joint position loss",
    "kobj": "Weight of the joint visibility loss, which drives joint confidence",
    "hsv_h": "Random hue change during training",
    "hsv_s": "Random saturation change during training",
    "hsv_v": "Random brightness change during training",
    "degrees": "Random rotation during training. 0, cameras are mounted level",
    "translate": "Random shift during training",
    "scale": "Random zoom during training. Mild, the service crop size is fixed by the person box",
    "fliplr": "Chance of a left-right flip. Joints are swapped with flip_idx",
    "flipud": "Chance of an upside-down flip. 0, a camera never sees people upside down",
    "mosaic": "Chance of combining four images into one. 0, the service sends one worker per crop",
    "close_mosaic": "Epochs at the end without mosaic. Unused when mosaic is 0",
}

hparam_rows = []
for name, value in BASE_HPARAMS.items():
    hparam_rows.append({"hyperparameter": name, "value": value, "meaning": HPARAM_MEANING[name]})
save_table(pd.DataFrame(hparam_rows).set_index("hyperparameter"), "hyperparameters_base")

### The service-matching score

Ultralytics reports pose mAP, which compares joints with OKS (object keypoint similarity) over many thresholds. That is a good general score, but it does not ask what the service asks. So every model is also scored by `service_scores`, which repeats the live steps on each crop:

1. Predict at `LIVE_CONF` and `IMGSZ`.
2. Keep the highest-scoring pose that has a head joint or a shoulder above `KP_MIN_CONF` (`_joints_inside`). No pose at all means the worker is kept without joints; poses that all fail the rule mean the worker is dropped.
3. Compare that pose with the labels of the crop's own person.

| Score | Meaning |
|-------|---------|
| `service_pck` | Share of labelled joints that the service would accept (confidence at least `KP_MIN_CONF`) and that lie within `PCK_ALPHA` of the box's longer side from the label. This is the selection score |
| `pck <group>` | The same, for one joint group (head, shoulders, wrists, ...) |
| `kept_rate` | Share of crops where the worker is not dropped |
| `joints_rate` | Share of crops where joints are placed |
| `false_joint_rate` | Share of unlabelled joints that the service would still accept. These put PPE regions in the wrong place |
| `stature_rate` | Of the crops whose labels allow a head-to-ankles stature, the share where the prediction allows it too |
| `stature_rel_error` | Median relative error of that predicted stature, which feeds distance estimation |

In [ ]:
def has_head_or_shoulders(points):
    """Same rule as ppe_inference.worker_has_head_or_shoulders."""
    if any(points[index, 2] >= KP_MIN_CONF for index in HEAD_IDX):
        return True
    return points[L_SHO, 2] >= KP_MIN_CONF or points[R_SHO, 2] >= KP_MIN_CONF


def stature_px(points):
    """Same rule as distance_estimation.stature_px: head (nose, or both eyes) to both ankles."""
    def confident(index):
        return points[index, 2] >= KP_MIN_CONF

    if confident(NOSE):
        head_y = points[NOSE, 1]
    elif confident(L_EYE) and confident(R_EYE):
        head_y = (points[L_EYE, 1] + points[R_EYE, 1]) * 0.5
    else:
        return None
    if not (confident(L_ANK) and confident(R_ANK)):
        return None
    height = (points[L_ANK, 1] + points[R_ANK, 1]) * 0.5 - head_y
    if height < 10.0:
        return None
    return float(height)


def predict_crops(weights, images, imgsz=IMGSZ, conf=None):
    """Raw pose predictions per image: joints (N, 17, 3) with confidence, and person scores."""
    if conf is None:
        conf = LIVE_CONF
    model = YOLO(str(weights))
    predictions = []
    for start in range(0, len(images), 32):
        chunk = [str(path) for path in images[start:start + 32]]
        results = model.predict(chunk, conf=conf, imgsz=imgsz, device=YOLO_DEVICE, verbose=False)
        for result in results:
            keypoints = result.keypoints
            if keypoints is None or len(keypoints) == 0 or result.boxes is None:
                predictions.append(None)
                continue
            xy = keypoints.xy.cpu().numpy()
            if keypoints.conf is not None:
                joint_conf = keypoints.conf.cpu().numpy()
            else:
                joint_conf = np.ones(xy.shape[:2])
            points = np.concatenate([xy, joint_conf[..., None]], axis=2)
            scores = result.boxes.conf.cpu().numpy()
            predictions.append({"points": points, "scores": scores})
    del model
    free_memory()
    return predictions


def pick_service_pose(prediction):
    """Which pose the service would keep, and what happens to the worker."""
    if prediction is None:
        return None, "no pose"
    best = None
    for index, score in enumerate(prediction["scores"]):
        points = prediction["points"][index]
        if not has_head_or_shoulders(points):
            continue
        if best is None or score > best[0]:
            best = (float(score), points)
    if best is None:
        return None, "dropped"
    return best[1], "joints placed"


def score_predictions(images, predictions):
    """Per-crop and per-joint results of the service rules against the crop's own person."""
    crop_rows = []
    joint_rows = []
    for image_path, prediction in zip(images, predictions):
        persons = read_pose_labels(label_path_for(image_path))
        with Image.open(image_path) as opened:
            width, height = opened.size
        main = persons[0]
        truth = labels_to_pixels(main, width, height)
        labelled = main["kpts"][:, 2] > 0
        scale = max(main["box"][2] * width, main["box"][3] * height)
        picked, outcome = pick_service_pose(prediction)
        if picked is not None:
            joint_conf = picked[:, 2]
            distance = np.hypot(picked[:, 0] - truth[:, 0], picked[:, 1] - truth[:, 1])
            close = distance <= PCK_ALPHA * scale
        else:
            joint_conf = np.zeros(NUM_KEYPOINTS)
            distance = np.full(NUM_KEYPOINTS, np.nan)
            close = np.zeros(NUM_KEYPOINTS, dtype=bool)
        accepted = joint_conf >= KP_MIN_CONF
        correct = labelled & accepted & close
        false_joint = (~labelled) & accepted

        truth_points = truth.copy()
        truth_points[:, 2] = labelled.astype(float)
        row = {
            "image": str(image_path),
            "outcome": outcome,
            "labelled": int(labelled.sum()),
            "unlabelled": int((~labelled).sum()),
            "correct": int(correct.sum()),
            "false_joints": int(false_joint.sum()),
            "true_stature": stature_px(truth_points),
            "pred_stature": stature_px(picked) if picked is not None else None,
        }
        for group_name, indices in JOINT_GROUPS.items():
            row[f"labelled {group_name}"] = int(labelled[indices].sum())
            row[f"correct {group_name}"] = int(correct[indices].sum())
        crop_rows.append(row)
        for index in range(NUM_KEYPOINTS):
            joint_rows.append({
                "image": str(image_path),
                "joint": KEYPOINT_NAMES[index],
                "group": JOINT_GROUP_OF[index],
                "labelled": bool(labelled[index]),
                "conf": float(joint_conf[index]),
                "close": bool(close[index]),
                "error_rel": float(distance[index] / scale) if labelled[index] and picked is not None else np.nan,
            })
    return pd.DataFrame(crop_rows), pd.DataFrame(joint_rows)


def summarise_service(per_crop):
    labelled_total = max(int(per_crop["labelled"].sum()), 1)
    summary = {
        "service_pck": float(per_crop["correct"].sum() / labelled_total),
        "kept_rate": float((per_crop["outcome"] != "dropped").mean()),
        "joints_rate": float((per_crop["outcome"] == "joints placed").mean()),
        "false_joint_rate": float(per_crop["false_joints"].sum() / max(int(per_crop["unlabelled"].sum()), 1)),
    }
    for group_name in JOINT_GROUPS:
        group_total = max(int(per_crop[f"labelled {group_name}"].sum()), 1)
        summary[f"pck {group_name}"] = float(per_crop[f"correct {group_name}"].sum() / group_total)
    has_truth = per_crop["true_stature"].notna()
    both = has_truth & per_crop["pred_stature"].notna()
    summary["stature_rate"] = float(both.sum() / max(int(has_truth.sum()), 1))
    if both.any():
        truth = per_crop.loc[both, "true_stature"].astype(float)
        pred = per_crop.loc[both, "pred_stature"].astype(float)
        summary["stature_rel_error"] = float(np.median(np.abs(pred - truth) / truth))
    else:
        summary["stature_rel_error"] = float("nan")
    return summary


def service_scores(weights, images, imgsz=IMGSZ, conf=None):
    predictions = predict_crops(weights, images, imgsz, conf)
    per_crop, per_joint = score_predictions(images, predictions)
    return summarise_service(per_crop), per_crop, per_joint

### Training and scoring functions

- `train_once` runs one Ultralytics training job.
- `score_model` runs Ultralytics validation on a split and returns its pose metrics.
- `run_experiment` trains one experiment (single or two stage), scores the best weights on val with both Ultralytics and `service_scores`, and records one row in `LEDGER`.
- `record_stock_model` scores stock weights without training, so they compete in the same ledger.

In [ ]:
LEDGER = []
EXPERIMENTS = {}


def resolve_weights(name):
    """Local path of weights given by name. Ultralytics downloads stock weights on first use."""
    model = YOLO(str(name))
    path = Path(getattr(model, "ckpt_path", None) or name).resolve()
    del model
    return path


def read_training_log(run_dir):
    results_csv = Path(run_dir) / "results.csv"
    if not results_csv.is_file():
        return pd.DataFrame()
    log = pd.read_csv(results_csv)
    log.columns = [column.strip() for column in log.columns]
    return log


def train_once(run_name, data_yaml, weights, imgsz, epochs, freeze, hparams, seed):
    model = YOLO(weights)
    args = dict(hparams)
    args["data"] = str(data_yaml)
    args["epochs"] = epochs
    args["patience"] = PATIENCE
    args["imgsz"] = imgsz
    args["batch"] = BATCH
    args["device"] = YOLO_DEVICE
    args["project"] = str(MODELS_DIR)
    args["name"] = run_name
    args["exist_ok"] = True
    args["seed"] = seed
    args["deterministic"] = True
    args["workers"] = 2
    args["plots"] = True
    args["freeze"] = freeze
    started = time.time()
    model.train(**args)
    elapsed = time.time() - started
    run_dir = Path(model.trainer.save_dir)
    best = run_dir / "weights" / "best.pt"
    del model
    free_memory()
    return run_dir, best, elapsed


def score_model(weights_path, data_yaml, imgsz, split="val", plots=False, name=None):
    model = YOLO(str(weights_path))
    kwargs = {
        "data": str(data_yaml),
        "split": split,
        "imgsz": imgsz,
        "batch": 16,
        "device": YOLO_DEVICE,
        "plots": plots,
        "verbose": False,
    }
    if name is not None:
        kwargs["project"] = str(RESULTS_DIR)
        kwargs["name"] = name
        kwargs["exist_ok"] = True
    metrics = model.val(**kwargs)
    del model
    free_memory()
    return metrics


def pose_metrics(metrics):
    return {
        "box_map50_95": float(metrics.box.map),
        "pose_precision": float(metrics.pose.mp),
        "pose_recall": float(metrics.pose.mr),
        "pose_map50": float(metrics.pose.map50),
        "pose_map50_95": float(metrics.pose.map),
    }


def best_epoch_of(log):
    if log.empty or "metrics/mAP50-95(P)" not in log.columns:
        return None
    return int(log.loc[log["metrics/mAP50-95(P)"].idxmax(), "epoch"])


SHOWN_COLUMNS = ["experiment_id", "change", "epochs_run", "pose_map50_95", "service_pck", "kept_rate",
                 "false_joint_rate", "stature_rate", "stature_rel_error", "minutes"]


def record(exp_id, summary, recipe, log, service, per_crop, per_joint):
    LEDGER.append(summary)
    EXPERIMENTS[exp_id] = {"summary": summary, "recipe": recipe, "log": log,
                           "service": service, "per_crop": per_crop, "per_joint": per_joint}
    display(pd.DataFrame([summary])[SHOWN_COLUMNS].set_index("experiment_id").round(3))


def record_stock_model(exp_id, change, weights):
    path = resolve_weights(weights)
    metrics = score_model(path, DATA_YAML, IMGSZ)
    service, per_crop, per_joint = service_scores(path, VAL_IMAGES)
    summary = {
        "experiment_id": exp_id, "group": "stock", "change": change, "weights_in": weights,
        "train images": 0, "imgsz": IMGSZ, "two_stage": False, "optimizer": "none", "seed": None,
        "epochs_run": 0, "best_epoch": None, "stopped_early": False,
        **pose_metrics(metrics), **service, "minutes": 0.0, "best_pt": str(path),
    }
    recipe = {"data_yaml": None, "weights": weights, "imgsz": IMGSZ, "two_stage": False, "hparams": None}
    record(exp_id, summary, recipe, pd.DataFrame(), service, per_crop, per_joint)
    return summary


def run_experiment(exp_id, change, data_yaml, weights="yolov8n-pose.pt", imgsz=IMGSZ,
                   two_stage=False, hparams=None, seed=SEED, group="main"):
    if hparams is None:
        hparams = BASE_HPARAMS

    logs = []
    if two_stage:
        run_dir_a, best_a, time_a = train_once(
            f"{exp_id}_stageA_frozen", data_yaml, weights, imgsz,
            STAGE_A_EPOCHS, FREEZE_LAYERS, hparams, seed,
        )
        stage_b_hparams = dict(hparams)
        stage_b_hparams["lr0"] = hparams["lr0"] / 10
        stage_b_hparams["warmup_epochs"] = 0.0
        stage_b_hparams["cos_lr"] = True
        run_dir, best, time_b = train_once(
            f"{exp_id}_stageB_unfrozen", data_yaml, str(best_a), imgsz,
            MAX_EPOCHS, None, stage_b_hparams, seed,
        )
        elapsed = time_a + time_b
        log_a = read_training_log(run_dir_a)
        log_a["stage"] = "A frozen"
        logs.append(log_a)
        log_b = read_training_log(run_dir)
        log_b["stage"] = "B unfrozen"
        logs.append(log_b)
    else:
        run_dir, best, elapsed = train_once(exp_id, data_yaml, weights, imgsz, MAX_EPOCHS, None, hparams, seed)
        log = read_training_log(run_dir)
        log["stage"] = "single"
        logs.append(log)

    full_log = pd.concat(logs, ignore_index=True)
    full_log["epoch overall"] = np.arange(1, len(full_log) + 1)
    last_log = logs[-1]

    metrics = score_model(best, data_yaml, imgsz)
    service, per_crop, per_joint = service_scores(best, VAL_IMAGES, imgsz)
    train_list = yaml.safe_load(Path(data_yaml).read_text())["train"]

    summary = {
        "experiment_id": exp_id, "group": group, "change": change, "weights_in": weights,
        "train images": len(read_list(train_list)), "imgsz": imgsz, "two_stage": two_stage,
        "optimizer": hparams["optimizer"], "seed": seed,
        "epochs_run": len(last_log), "best_epoch": best_epoch_of(last_log),
        "stopped_early": len(last_log) < MAX_EPOCHS,
        **pose_metrics(metrics), **service,
        "minutes": round(elapsed / 60, 1), "best_pt": str(best),
    }
    recipe = {"data_yaml": Path(data_yaml), "weights": weights, "imgsz": imgsz,
              "two_stage": two_stage, "hparams": dict(hparams)}
    with (Path(run_dir) / "experiment_summary.json").open("w") as handle:
        json.dump(summary, handle, indent=2, default=str)
    record(exp_id, summary, recipe, full_log, service, per_crop, per_joint)
    return summary


def score_of(exp_id, metric=SELECTION_METRIC):
    return EXPERIMENTS[exp_id]["summary"][metric]

## 8. Experiments

Each experiment changes one thing compared with the step before it:

| Id | Change | Question it answers |
|----|--------|---------------------|
| E0 | Stock `yolov8n-pose` and `yolov8s-pose`, no training | What SAMS gets today, and what a larger stock model would give |
| E1 | Fine-tune `yolov8n-pose` on crops, SGD | Does training on service-style crops help at all? |
| E2 | Two-stage fine-tuning (freeze 10, then unfreeze at lr0/10) | Is it safer to adapt the head first? |
| E3 | AdamW lr0=0.0005 vs SGD | Which optimizer wins? |
| E4 | Add camera-damage copies | Does damaged training data cost accuracy on clean crops? |
| E5 | `yolov8s-pose` with the winning recipe | Does a larger model help? |

All runs use early stopping (`patience=5`, `max_epochs=30`) and the same val crops. The primary score is val `service_pck`; ties within 0.005 go to higher pose mAP50-95.

### E0. Stock models, no training

In [ ]:
record_stock_model("pose_e0_stock_n", "Stock yolov8n-pose, as served today", "yolov8n-pose.pt")
record_stock_model("pose_e0_stock_s", "Stock yolov8s-pose, larger and slower", "yolov8s-pose.pt")

### E1. Fine-tune YOLOv8n-pose on crops

In [ ]:
run_experiment(
    "pose_e1_finetune",
    "Fine-tune yolov8n-pose on padded crops, SGD lr0=0.002",
    data_yaml=DATA_YAML,
)

stock_score = score_of("pose_e0_stock_n")
e1_score = score_of("pose_e1_finetune")
e1_table = pd.DataFrame([
    {"model": "stock yolov8n-pose", SELECTION_METRIC: stock_score, "pose_map50_95": score_of("pose_e0_stock_n", "pose_map50_95")},
    {"model": "fine-tuned (E1)", SELECTION_METRIC: e1_score, "pose_map50_95": score_of("pose_e1_finetune", "pose_map50_95")},
]).set_index("model")
save_table(e1_table.round(4), "experiment_stock_vs_finetune")

### E2. Two-stage fine-tuning: freeze 10 layers, then unfreeze at lr0/10

In [ ]:
run_experiment(
    "pose_e2_two_stage",
    f"Freeze {FREEZE_LAYERS} layers for {STAGE_A_EPOCHS} epochs, then unfreeze at lr0/10",
    data_yaml=DATA_YAML,
    two_stage=True,
)

USE_TWO_STAGE = score_of("pose_e2_two_stage") >= score_of("pose_e1_finetune")
if USE_TWO_STAGE:
    sgd_id = "pose_e2_two_stage"
else:
    sgd_id = "pose_e1_finetune"
FINAL_RECIPE = {"data_yaml": DATA_YAML, "two_stage": USE_TWO_STAGE, "hparams": dict(BASE_HPARAMS)}

fine_tune_table = pd.DataFrame([
    {"recipe": "single stage", "experiment": "pose_e1_finetune", SELECTION_METRIC: score_of("pose_e1_finetune")},
    {"recipe": "two stage", "experiment": "pose_e2_two_stage", SELECTION_METRIC: score_of("pose_e2_two_stage")},
]).set_index("recipe")
save_table(fine_tune_table.round(4), "experiment_two_stage_choice")
print("Two-stage kept:", USE_TWO_STAGE)

### E3. Optimizer: AdamW lr0=0.0005 vs SGD

In [ ]:
ADAMW_HPARAMS = dict(FINAL_RECIPE["hparams"])
ADAMW_HPARAMS["optimizer"] = "AdamW"
ADAMW_HPARAMS["lr0"] = 0.0005
ADAMW_HPARAMS["momentum"] = 0.9

run_experiment(
    "pose_e3_adamw",
    "AdamW lr0=0.0005 with the recipe so far",
    data_yaml=FINAL_RECIPE["data_yaml"],
    two_stage=FINAL_RECIPE["two_stage"],
    hparams=ADAMW_HPARAMS,
)

if score_of("pose_e3_adamw") > score_of(sgd_id):
    FINAL_RECIPE["hparams"] = ADAMW_HPARAMS
    recipe_id = "pose_e3_adamw"
else:
    recipe_id = sgd_id

optimizer_table = pd.DataFrame([
    {"optimizer": "SGD", "experiment": sgd_id, SELECTION_METRIC: score_of(sgd_id)},
    {"optimizer": "AdamW", "experiment": "pose_e3_adamw", SELECTION_METRIC: score_of("pose_e3_adamw")},
]).set_index("optimizer")
save_table(optimizer_table.round(4), "experiment_optimizer_choice")
print("Optimizer kept:", FINAL_RECIPE["hparams"]["optimizer"])

### E4. Add camera-damage copies

Val is clean, so it cannot reward robustness. Camera copies are therefore kept unless they cost more than `WINNER_TOLERANCE` on val; section 10.5 then checks whether they actually help on damaged crops.

In [ ]:
run_experiment(
    "pose_e4_camera",
    "Recipe so far plus camera-damage copies of every train crop",
    data_yaml=DATA_YAML_CAMERA,
    two_stage=FINAL_RECIPE["two_stage"],
    hparams=FINAL_RECIPE["hparams"],
)

USE_CAMERA = score_of("pose_e4_camera") >= score_of(recipe_id) - WINNER_TOLERANCE
if USE_CAMERA:
    FINAL_RECIPE["data_yaml"] = DATA_YAML_CAMERA

camera_table = pd.DataFrame([
    {"training data": "crops only", "experiment": recipe_id, SELECTION_METRIC: score_of(recipe_id)},
    {"training data": "crops plus camera copies", "experiment": "pose_e4_camera", SELECTION_METRIC: score_of("pose_e4_camera")},
]).set_index("training data")
save_table(camera_table.round(4), "experiment_camera_choice")
print("Camera copies kept:", USE_CAMERA)

### E5. YOLOv8s-pose with the winning recipe

In [ ]:
run_experiment(
    "pose_e5_yolov8s",
    "yolov8s-pose with the best fine-tuning, optimizer, and data so far",
    data_yaml=FINAL_RECIPE["data_yaml"],
    weights="yolov8s-pose.pt",
    two_stage=FINAL_RECIPE["two_stage"],
    hparams=FINAL_RECIPE["hparams"],
)

## 9. Compare the experiments and choose the model

The winner is the run with the highest val `service_pck`, stock models included. If another run is within `WINNER_TOLERANCE`, the one with the higher pose mAP50-95 wins instead. If a stock model wins, it is promoted unchanged: fine-tuning only replaces the stock model when it is measurably better.

In [ ]:
ledger_frame = pd.DataFrame(LEDGER)
ledger_columns = ["experiment_id", "group", "change", "weights_in", "train images", "two_stage", "optimizer",
                  "epochs_run", "best_epoch", "pose_map50", "pose_map50_95", "service_pck", "kept_rate",
                  "joints_rate", "false_joint_rate", "stature_rate", "stature_rel_error", "minutes"]
ledger_view = ledger_frame[ledger_columns].set_index("experiment_id").round(4)
save_table(ledger_view, "experiment_ledger")

candidates = ledger_frame[ledger_frame["group"].isin(["main", "stock"])].reset_index(drop=True)
fig, axes = plt.subplots(1, 2, figsize=(17, 5))
positions = np.arange(len(candidates))
bar_width = 0.25
for offset, (column, label, color) in enumerate([
    ("pose_map50_95", "pose mAP50-95", "#2b6cb0"),
    ("service_pck", "service PCK", "#2f855a"),
    ("kept_rate", "kept rate", "#d69e2e"),
]):
    axes[0].bar(positions + (offset - 1) * bar_width, candidates[column], bar_width, label=label, color=color)
axes[0].set_xticks(positions)
axes[0].set_xticklabels(candidates["experiment_id"], rotation=30, ha="right")
axes[0].set_ylim(0, 1)
axes[0].set_title("Val scores per experiment")
axes[0].legend()

group_columns = [f"pck {group_name}" for group_name in JOINT_GROUPS]
group_view = candidates.set_index("experiment_id")[group_columns]
group_view.columns = list(JOINT_GROUPS)
sns.heatmap(group_view, annot=True, fmt=".2f", cmap="Greens", vmin=0, vmax=1, ax=axes[1])
axes[1].set_title("Val PCK per joint group")
plt.tight_layout()
save_figure(fig, "experiment_comparison")

### Training curves

Val pose mAP50-95 and val pose loss after every epoch. A line that flattens long before it ends shows where early stopping stepped in. A val loss that rises while training continues is overfitting. For two-stage runs, stage A and stage B are joined, so the jump where the backbone is unfrozen is visible. Stock models have no curve.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
for exp_id, experiment in EXPERIMENTS.items():
    if experiment["summary"]["group"] != "main":
        continue
    log = experiment["log"]
    if "metrics/mAP50-95(P)" in log.columns:
        axes[0].plot(log["epoch overall"], log["metrics/mAP50-95(P)"], label=exp_id)
    if "val/pose_loss" in log.columns:
        axes[1].plot(log["epoch overall"], log["val/pose_loss"], label=exp_id)
for exp_id in ["pose_e0_stock_n"]:
    axes[0].axhline(score_of(exp_id, "pose_map50_95"), color="grey", linestyle="--", label=f"{exp_id} (no training)")
axes[0].set_title("Val pose mAP50-95 per epoch")
axes[0].set_xlabel("epoch")
axes[1].set_title("Val pose loss per epoch")
axes[1].set_xlabel("epoch")
axes[0].legend(fontsize=8)
axes[1].legend(fontsize=8)
plt.tight_layout()
save_figure(fig, "training_curves")

In [ ]:
ranked = candidates.sort_values(SELECTION_METRIC, ascending=False).reset_index(drop=True)
top_score = ranked.loc[0, SELECTION_METRIC]
close_runs = ranked[ranked[SELECTION_METRIC] >= top_score - WINNER_TOLERANCE]
winner_row = close_runs.sort_values("pose_map50_95", ascending=False).iloc[0]

WINNER_ID = winner_row["experiment_id"]
WINNER = EXPERIMENTS[WINNER_ID]
WINNER_IS_STOCK = WINNER["summary"]["group"] == "stock"
BEST_WEIGHTS = Path(WINNER["summary"]["best_pt"])
STOCK_ID = "pose_e0_stock_n"
STOCK_WEIGHTS = Path(EXPERIMENTS[STOCK_ID]["summary"]["best_pt"])

winner_table = pd.DataFrame([{
    "chosen experiment": WINNER_ID,
    "stock model": WINNER_IS_STOCK,
    "val service PCK": winner_row["service_pck"],
    "val pose mAP50-95": winner_row["pose_map50_95"],
    "gain over stock yolov8n-pose (PCK)": winner_row["service_pck"] - score_of(STOCK_ID),
    "runs within tolerance": len(close_runs),
}]).set_index("chosen experiment")
save_table(winner_table.round(4), "winner")

### Repeat with other seeds

A single training run has some luck in it: the order of the images, the augmentation draws, and the starting state of training. The winning recipe is trained again with each seed in `REPEAT_SEEDS`. The standard deviation of the runs shows how big a gap between experiments has to be before it means anything. The second table compares each experiment's gap to the winner with that standard deviation. If a stock model won, there is no training to repeat and the cell says so.

In [ ]:
if WINNER_IS_STOCK:
    display(pd.DataFrame([{"result": f"{WINNER_ID} is a stock model, so there is no training run to repeat"}]))
    SEED_STD = float("nan")
else:
    repeat_ids = [WINNER_ID]
    for repeat_seed in REPEAT_SEEDS:
        repeat_id = f"{WINNER_ID}_seed{repeat_seed}"
        run_experiment(
            repeat_id,
            f"Repeat of {WINNER_ID} with seed {repeat_seed}",
            data_yaml=WINNER["recipe"]["data_yaml"],
            weights=WINNER["recipe"]["weights"],
            imgsz=WINNER["recipe"]["imgsz"],
            two_stage=WINNER["recipe"]["two_stage"],
            hparams=WINNER["recipe"]["hparams"],
            seed=repeat_seed,
            group="seed repeat",
        )
        repeat_ids.append(repeat_id)

    score_columns = ["pose_map50_95", "service_pck", "kept_rate", "false_joint_rate", "stature_rel_error"]
    repeat_frame = pd.DataFrame([EXPERIMENTS[exp_id]["summary"] for exp_id in repeat_ids]).set_index("experiment_id")
    spread = repeat_frame[score_columns].agg(["mean", "std"])
    save_table(pd.concat([repeat_frame[score_columns], spread]).round(4), "seed_repeats")
    SEED_STD = float(spread.loc["std", SELECTION_METRIC])

    gap_rows = []
    for _, row in candidates.iterrows():
        gap = score_of(WINNER_ID) - row[SELECTION_METRIC]
        gap_rows.append({
            "experiment": row["experiment_id"],
            f"gap to winner ({SELECTION_METRIC})": gap,
            "gap larger than seed std": bool(gap > SEED_STD) if row["experiment_id"] != WINNER_ID else None,
        })
    save_table(pd.DataFrame(gap_rows).set_index("experiment").round(4), "seed_gap_check")

## 10. Evaluate the chosen model in detail

Everything in this section uses val only, and compares the chosen model with the stock `yolov8n-pose` that SAMS serves today. Test is still untouched.

### 10.1 Joint groups and what happens to each worker

The left chart is PCK per joint group. Wrists and ankles are usually the weakest, and they are exactly where the service looks for gloves and safety shoes. The right chart splits the crops by what the service would do: place joints, keep the worker without joints, or drop the worker as a false person. Every val crop holds a real labelled person, so a dropped worker here is always a missed worker.

In [ ]:
compare_ids = [STOCK_ID] if WINNER_ID == STOCK_ID else [STOCK_ID, WINNER_ID]
group_rows = []
outcome_rows = []
for exp_id in compare_ids:
    service = EXPERIMENTS[exp_id]["service"]
    for group_name in JOINT_GROUPS:
        group_rows.append({"model": exp_id, "group": group_name, "PCK": service[f"pck {group_name}"]})
    counts = EXPERIMENTS[exp_id]["per_crop"]["outcome"].value_counts(normalize=True) * 100
    outcome_rows.append({
        "model": exp_id,
        "joints placed %": counts.get("joints placed", 0.0),
        "kept without joints %": counts.get("no pose", 0.0),
        "dropped %": counts.get("dropped", 0.0),
    })
group_frame = pd.DataFrame(group_rows)
outcome_frame = pd.DataFrame(outcome_rows).set_index("model")
save_table(group_frame.pivot(index="group", columns="model", values="PCK").reindex(list(JOINT_GROUPS)).round(3), "best_val_pck_by_group")
save_table(outcome_frame.round(1), "best_val_outcomes")

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
sns.barplot(data=group_frame, x="group", y="PCK", hue="model", order=list(JOINT_GROUPS), ax=axes[0])
axes[0].set_ylim(0, 1)
axes[0].set_title("Val PCK per joint group")
outcome_frame.plot(kind="barh", stacked=True, ax=axes[1], color=["#2f855a", "#d69e2e", "#c53030"])
axes[1].set_xlabel("percent of val crops")
axes[1].set_title("What the service does with each worker")
plt.tight_layout()
save_figure(fig, "best_val_groups_and_outcomes")

### 10.2 Stature for distance estimation

Distance estimation uses the head-to-ankles stature only when the nose (or both eyes) and both ankles are confident. The scatter compares predicted and labelled stature where both exist; points on the diagonal are perfect. The table gives how often stature was available and its median relative error.

In [ ]:
stature_rows = []
fig, axes = plt.subplots(1, len(compare_ids), figsize=(7 * len(compare_ids), 6), squeeze=False)
for ax, exp_id in zip(axes[0], compare_ids):
    per_crop = EXPERIMENTS[exp_id]["per_crop"]
    both = per_crop["true_stature"].notna() & per_crop["pred_stature"].notna()
    truth = per_crop.loc[both, "true_stature"].astype(float)
    pred = per_crop.loc[both, "pred_stature"].astype(float)
    ax.scatter(truth, pred, s=8, alpha=0.5)
    limit = float(max(truth.max(), pred.max())) if both.any() else 1.0
    ax.plot([0, limit], [0, limit], color="grey", linestyle="--")
    ax.set_xlabel("labelled stature (pixels)")
    ax.set_ylabel("predicted stature (pixels)")
    ax.set_title(exp_id)
    relative = (pred - truth).abs() / truth
    stature_rows.append({
        "model": exp_id,
        "crops where labels allow stature": int(per_crop["true_stature"].notna().sum()),
        "share where prediction allows it": EXPERIMENTS[exp_id]["service"]["stature_rate"],
        "median relative error": float(relative.median()) if both.any() else float("nan"),
        "90th percentile relative error": float(relative.quantile(0.9)) if both.any() else float("nan"),
    })
plt.tight_layout()
save_figure(fig, "best_val_stature")
save_table(pd.DataFrame(stature_rows).set_index("model").round(3), "best_val_stature")

### 10.3 Is the service's joint threshold right for this model?

The service accepts a joint at confidence 0.3, whatever the model. For thresholds from 0.05 to 0.9, the chart shows two shares for the chosen model:

- labelled joints that are accepted and correct (higher is better),
- unlabelled joints that are still accepted (lower is better; these place PPE regions on nothing).

Fine-tuning can shift joint confidences up or down. If the best balance is far from 0.3, the suggested value is written to `metadata.json` as `kpt_conf_suggested`, so you can decide whether to change `_KP_MIN`. The notebook does not change the service.

In [ ]:
winner_joints = EXPERIMENTS[WINNER_ID]["per_joint"]
labelled_joints_w = winner_joints[winner_joints["labelled"]]
unlabelled_joints_w = winner_joints[~winner_joints["labelled"]]
threshold_rows = []
for threshold in np.round(np.arange(0.05, 0.91, 0.05), 2):
    correct_share = float(((labelled_joints_w["conf"] >= threshold) & labelled_joints_w["close"]).mean())
    false_share = float((unlabelled_joints_w["conf"] >= threshold).mean()) if len(unlabelled_joints_w) else 0.0
    threshold_rows.append({"threshold": threshold, "correct labelled %": correct_share * 100,
                           "accepted unlabelled %": false_share * 100, "balance": correct_share - false_share})
threshold_frame = pd.DataFrame(threshold_rows).set_index("threshold")
KPT_CONF_SUGGESTED = float(threshold_frame["balance"].idxmax())
save_table(threshold_frame.round(2), "best_val_joint_threshold_sweep")

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(threshold_frame.index, threshold_frame["correct labelled %"], label="labelled joints accepted and correct")
ax.plot(threshold_frame.index, threshold_frame["accepted unlabelled %"], label="unlabelled joints accepted")
ax.axvline(KP_MIN_CONF, color="grey", linestyle="--", label=f"service threshold {KP_MIN_CONF}")
ax.axvline(KPT_CONF_SUGGESTED, color="#c53030", linestyle=":", label=f"best balance {KPT_CONF_SUGGESTED}")
ax.set_xlabel("joint confidence threshold")
ax.set_ylabel("percent")
ax.set_title(f"Joint threshold sweep, {WINNER_ID}")
ax.legend(fontsize=8)
plt.tight_layout()
save_figure(fig, "best_val_joint_threshold_sweep")

### 10.4 Bootstrap confidence intervals

The val crops are resampled with replacement `BOOTSTRAP_ROUNDS` times, keeping crops of one source photo together, and the PCK of both models and their difference are recomputed each time. The 2.5 and 97.5 percentiles give a 95 percent interval. If the interval of the difference includes 0, the chosen model is not reliably better than the stock one.

In [ ]:
def bootstrap_pck(per_crop_a, per_crop_b, images, rounds=BOOTSTRAP_ROUNDS):
    source_of = manifest.set_index("image")["source"]
    groups = pd.Series([source_of.get(str(path), str(path)) for path in images])
    unique_sources = groups.unique()
    rows_by_source = {source: np.where(groups.to_numpy() == source)[0] for source in unique_sources}
    rng = np.random.default_rng(SEED)
    a_correct = per_crop_a["correct"].to_numpy()
    b_correct = per_crop_b["correct"].to_numpy()
    labelled = per_crop_a["labelled"].to_numpy()
    results = []
    for _ in range(rounds):
        chosen = rng.choice(unique_sources, size=len(unique_sources), replace=True)
        rows = np.concatenate([rows_by_source[source] for source in chosen])
        total = max(labelled[rows].sum(), 1)
        pck_a = a_correct[rows].sum() / total
        pck_b = b_correct[rows].sum() / total
        results.append((pck_a, pck_b, pck_b - pck_a))
    results = np.array(results)
    return pd.DataFrame({
        "stock PCK": results[:, 0], "chosen PCK": results[:, 1], "difference": results[:, 2],
    }).quantile([0.025, 0.5, 0.975])


val_intervals = bootstrap_pck(EXPERIMENTS[STOCK_ID]["per_crop"], WINNER["per_crop"], VAL_IMAGES)
val_intervals.index = ["2.5%", "median", "97.5%"]
save_table(val_intervals.round(4), "best_val_bootstrap_intervals")

### 10.5 Robustness to bad camera conditions

Every val crop is scored as it is, and again after one fixed damage at a time. Only the pixels change, so the labels are the same. These settings are stronger than the camera copies used in training.

| Condition | What it imitates |
|-----------|------------------|
| motion blur | A worker moving fast, or camera shake |
| dark | Night shift or a badly lit corner |
| heavy jpeg | A heavily compressed stream |
| low resolution | A distant worker on a low-resolution camera |
| sensor noise | A cheap or overheated sensor |

A large drop shows which condition needs more training data, and whether the camera copies in E4 helped.

In [ ]:
CONDITIONS = {
    "clean": None,
    "motion blur": A.MotionBlur(blur_limit=(9, 13), p=1.0),
    "dark": A.RandomBrightnessContrast(brightness_limit=(-0.45, -0.35), contrast_limit=(0.0, 0.0), p=1.0),
    "heavy jpeg": A.ImageCompression(quality_range=(10, 20), p=1.0),
    "low resolution": A.Downscale(scale_range=(0.3, 0.35), p=1.0),
    "sensor noise": A.GaussNoise(std_range=(0.08, 0.12), p=1.0),
}

robust_rows = []
for condition, transform in CONDITIONS.items():
    if transform is None:
        condition_images = VAL_IMAGES
    else:
        folder = "robust_" + condition.replace(" ", "_")
        image_dir = MERGED / "images" / folder
        if image_dir.exists():
            shutil.rmtree(image_dir)
        image_dir.mkdir(parents=True)
        seeded = A.Compose([transform], seed=SEED)
        condition_images = []
        for image_path in VAL_IMAGES:
            with Image.open(image_path) as opened:
                damaged = seeded(image=np.asarray(opened.convert("RGB")))["image"]
            target = image_dir / image_path.name
            Image.fromarray(damaged).save(target, quality=95)
            label_target = label_path_for(target)
            label_target.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(label_path_for(image_path), label_target)
            condition_images.append(target)
    for exp_id in compare_ids:
        weights = EXPERIMENTS[exp_id]["summary"]["best_pt"]
        service, _, _ = service_scores(weights, condition_images)
        robust_rows.append({"condition": condition, "model": exp_id, "service PCK": service["service_pck"],
                            "kept rate": service["kept_rate"], "stature rate": service["stature_rate"]})

robust_frame = pd.DataFrame(robust_rows)
robust_table = robust_frame.pivot(index="condition", columns="model", values=["service PCK", "kept rate"])
robust_table = robust_table.reindex(list(CONDITIONS))
save_table(robust_table.round(3), "best_val_robustness")

fig, ax = plt.subplots(figsize=(11, 4.5))
sns.barplot(data=robust_frame, x="condition", y="service PCK", hue="model", order=list(CONDITIONS), ax=ax)
ax.set_ylim(0, 1)
ax.set_title("Val service PCK under camera damage")
plt.tight_layout()
save_figure(fig, "best_val_robustness")

### 10.6 Where the chosen model gets joints wrong

The six val crops with the lowest PCK among crops with at least 8 labelled joints. Green is the label, red is the pose the service would keep. Look for a pattern: a neighbour's pose kept instead of the worker's, left and right swapped, hands hidden behind objects, or a label that is itself wrong.

In [ ]:
per_crop_w = WINNER["per_crop"].copy()
per_crop_w["pck"] = per_crop_w["correct"] / per_crop_w["labelled"].clip(lower=1)
worst = per_crop_w[per_crop_w["labelled"] >= 8].sort_values("pck").head(6)

if worst.empty:
    display(pd.DataFrame([{"result": "No val crop with at least 8 labelled joints"}]))
else:
    worst_images = [Path(path) for path in worst["image"]]
    worst_predictions = predict_crops(BEST_WEIGHTS, worst_images)
    fig, axes = plt.subplots(2, 3, figsize=(15, 11))
    for ax in axes.flatten():
        ax.axis("off")
    for ax, image_path, prediction, (_, row) in zip(axes.flatten(), worst_images, worst_predictions, worst.iterrows()):
        persons = read_pose_labels(label_path_for(image_path))
        with Image.open(image_path) as opened:
            image = opened.convert("RGB")
        ax.imshow(image)
        draw_skeleton(ax, labels_to_pixels(persons[0], image.width, image.height), color="#2f855a")
        picked, outcome = pick_service_pose(prediction)
        if picked is not None:
            draw_skeleton(ax, picked, color="#e53e3e", min_value=KP_MIN_CONF)
        ax.set_title(f"PCK {row['pck']:.2f}, {outcome}, {len(persons)} labelled people", fontsize=9)
    fig.suptitle(f"Hardest val crops for {WINNER_ID} (green label, red prediction)")
    plt.tight_layout()
    save_figure(fig, "best_val_worst_crops")

## 11. Final test, live metadata, export check, model card

### 11.1 Score the chosen model once on test

The chosen weights are copied to `models/vision/pose/best.pt` and scored on the test split, next to the stock `yolov8n-pose`. This is the only time test is used. Two views are reported: the Ultralytics pose metrics, and the service-matching scores with a bootstrap interval on the PCK gain.

In [ ]:
PROMOTED = MODELS_DIR / "best.pt"
shutil.copy2(BEST_WEIGHTS, PROMOTED)

test_rows = []
TEST_RESULTS = {}
for label, weights in [("stock yolov8n-pose", STOCK_WEIGHTS), ("chosen", PROMOTED)]:
    metrics = pose_metrics(score_model(weights, DATA_YAML, IMGSZ, split="test"))
    service, per_crop, _ = service_scores(weights, TEST_IMAGES)
    TEST_RESULTS[label] = {**metrics, **service, "per_crop": per_crop}
    test_rows.append({"model": label, **metrics, **service})
test_table = pd.DataFrame(test_rows).set_index("model")
save_table(test_table.round(4), "test_scores")

test_intervals = bootstrap_pck(TEST_RESULTS["stock yolov8n-pose"]["per_crop"], TEST_RESULTS["chosen"]["per_crop"], TEST_IMAGES)
test_intervals.index = ["2.5%", "median", "97.5%"]
save_table(test_intervals.round(4), "test_bootstrap_intervals")

### 11.2 Metadata for the live service

`vision_inference.py` reads `model_id` (shown in the Vision lab model tag) and `imgsz` (passed to `predict`) from this file. The other keys are for people and tooling: the keypoint layout, the operating points the scores were measured at, the training recipe, and the val and test scores.

In [ ]:
promoted_model = YOLO(str(PROMOTED))
promoted_names = {int(class_id): str(name) for class_id, name in promoted_model.names.items()}
kpt_shape = getattr(promoted_model.model, "kpt_shape", None) or promoted_model.model.yaml.get("kpt_shape")
kpt_shape = [int(value) for value in kpt_shape]
del promoted_model

train_hash = hashlib.sha256(TRAIN_LIST.read_bytes()).hexdigest()
if WINNER_IS_STOCK:
    model_id = Path(WINNER["recipe"]["weights"]).stem
else:
    model_id = f"{Path(WINNER['recipe']['weights']).stem}_sams"

recipe_hparams = WINNER["recipe"]["hparams"]
metadata = {
    "model_id": model_id,
    "best_experiment_id": WINNER_ID,
    "stock_model": WINNER_IS_STOCK,
    "trained_utc": datetime.now(timezone.utc).isoformat(),
    "imgsz": IMGSZ,
    "kpt_shape": kpt_shape,
    "keypoint_names": KEYPOINT_NAMES,
    "flip_idx": FLIP_IDX,
    "class_names": promoted_names,
    "operating_point": {"person_conf": LIVE_CONF, "kpt_conf": KP_MIN_CONF, "crop_pad": CROP_PAD},
    "kpt_conf_suggested": KPT_CONF_SUGGESTED,
    "dataset": {
        "name": "COCO 2017 val keypoints, one padded crop per person",
        "labels_url": COCO_LABELS_URL,
        "images_url": COCO_IMAGES_URL,
        "license": "COCO annotations CC BY 4.0, images under their Flickr licenses",
        "source_photos": int(manifest["source"].nunique()),
        "train_crops": len(TRAIN_IMAGES),
        "val_crops": len(VAL_IMAGES),
        "test_crops": len(TEST_IMAGES),
        "site_crops": len(SITE_IMAGES),
        "camera_copies_used": bool(WINNER["recipe"]["data_yaml"] == DATA_YAML_CAMERA),
        "train_list_sha256": train_hash,
    },
    "recipe": {
        "weights_in": WINNER["recipe"]["weights"],
        "two_stage": WINNER["recipe"]["two_stage"],
        "freeze_layers_stage_a": FREEZE_LAYERS,
        "patience": PATIENCE,
        "hyperparameters": recipe_hparams,
    },
    "val": {key: WINNER["summary"][key] for key in ["pose_map50", "pose_map50_95", "service_pck", "kept_rate",
                                                     "false_joint_rate", "stature_rate", "stature_rel_error"]},
    "test": {key: TEST_RESULTS["chosen"][key] for key in ["pose_map50", "pose_map50_95", "service_pck", "kept_rate",
                                                         "false_joint_rate", "stature_rate", "stature_rel_error"]},
    "test_stock_yolov8n_pose": {key: TEST_RESULTS["stock yolov8n-pose"][key] for key in ["pose_map50_95", "service_pck", "kept_rate"]},
}
METADATA_PATH = MODELS_DIR / "metadata.json"
METADATA_PATH.write_text(json.dumps(metadata, indent=2, default=str))
print("Wrote", METADATA_PATH)

The file is read back and checked the way the service relies on it: the weights must have 17 joints of three values, the keypoint names at every index the service uses must be the expected joints, and `imgsz` must be a positive number. If any check fails, the notebook stops instead of shipping a model the service would read wrongly.

In [ ]:
spec = json.loads(METADATA_PATH.read_text())
check_rows = []
if spec["kpt_shape"] != [NUM_KEYPOINTS, 3]:
    raise RuntimeError(f"Weights have kpt_shape {spec['kpt_shape']}, the service needs [17, 3]")
check_rows.append({"check": "kpt_shape is [17, 3]", "result": "ok"})
for index, expected in SERVICE_EXPECTED_NAMES.items():
    if spec["keypoint_names"][index] != expected:
        raise RuntimeError(f"Joint {index} is {spec['keypoint_names'][index]}, the service expects {expected}")
check_rows.append({"check": "joint order matches the service indices", "result": "ok"})
if int(spec["imgsz"]) <= 0:
    raise RuntimeError("imgsz must be positive")
check_rows.append({"check": "imgsz", "result": str(spec["imgsz"])})
check_rows.append({"check": "model_id", "result": spec["model_id"]})
check_rows.append({"check": "weights file", "result": f"{PROMOTED} ({PROMOTED.stat().st_size / 1e6:.1f} MB)"})
save_table(pd.DataFrame(check_rows).set_index("check"), "metadata_checks")

### 11.3 Export check

The model is exported to ONNX, the format most deployment runtimes accept. Both versions are timed on `TIMING_IMAGES` single-crop predictions, the way the live stream runs pose once per worker. Val crops are cycled to reach that number. Without `onnxruntime-gpu`, the ONNX model runs on the CPU, so its speed shows CPU deployment. The live service loads `best.pt`, so ONNX is a deployment option, not a requirement.

In [ ]:
def measure_fps(weights_path, images, imgsz):
    model = YOLO(str(weights_path), task="pose")
    for image_path in images[:5]:
        model.predict(str(image_path), imgsz=imgsz, device=YOLO_DEVICE, verbose=False)
    started = time.time()
    for image_path in images:
        model.predict(str(image_path), imgsz=imgsz, device=YOLO_DEVICE, verbose=False)
    elapsed = time.time() - started
    del model
    free_memory()
    return len(images) / elapsed


export_model = YOLO(str(PROMOTED))
onnx_path = Path(export_model.export(format="onnx", imgsz=IMGSZ, simplify=True))
del export_model

timing_images = [VAL_IMAGES[number % len(VAL_IMAGES)] for number in range(TIMING_IMAGES)]
onnx_service, _, _ = service_scores(onnx_path, TEST_IMAGES)
export_table = pd.DataFrame([
    {"format": "PyTorch (.pt)", "file": PROMOTED.name, "test service PCK": TEST_RESULTS["chosen"]["service_pck"],
     "crops per second": measure_fps(PROMOTED, timing_images, IMGSZ)},
    {"format": "ONNX", "file": onnx_path.name, "test service PCK": onnx_service["service_pck"],
     "crops per second": measure_fps(onnx_path, timing_images, IMGSZ)},
    {"format": "stock yolov8n-pose (.pt)", "file": STOCK_WEIGHTS.name,
     "test service PCK": TEST_RESULTS["stock yolov8n-pose"]["service_pck"],
     "crops per second": measure_fps(STOCK_WEIGHTS, timing_images, IMGSZ)},
]).set_index("format")
save_table(export_table.round(3), "export_check")

### 11.4 Model card

A one-page summary saved as `model_card.md` next to the weights, so anyone using the model can see where it came from, how well it works, and where it is weak.

In [ ]:
stock_test = TEST_RESULTS["stock yolov8n-pose"]
chosen_test = TEST_RESULTS["chosen"]
card_lines = [
    "# SAMS worker pose model",
    "",
    f"- Model: {metadata['model_id']} (experiment {WINNER_ID})",
    f"- Stock model promoted unchanged: {WINNER_IS_STOCK}",
    f"- Trained: {metadata['trained_utc']}",
    f"- Image size: {IMGSZ}",
    f"- Training crops: {len(TRAIN_IMAGES)} COCO plus {len(SITE_IMAGES)} site, val crops: {len(VAL_IMAGES)}, test crops: {len(TEST_IMAGES)}",
    "",
    "## Intended use",
    "",
    "Places 17 COCO body joints on one worker inside a padded crop of the PPE model's person box, in",
    "`monitoring/services/vision_inference.py`. The joints decide where PPE counts as worn, reject false",
    "people without head or shoulders, and give the head-to-ankles stature used for distance.",
    "",
    "## Scores on test (service rules: person conf "
    f"{LIVE_CONF}, joint conf {KP_MIN_CONF}, PCK at {PCK_ALPHA} of the box)",
    "",
    "| Score | Stock yolov8n-pose | This model |",
    "|-------|--------------------|------------|",
]
for key in ["pose_map50_95", "service_pck", "kept_rate", "false_joint_rate", "stature_rate", "stature_rel_error"]:
    card_lines.append(f"| {key} | {stock_test[key]:.3f} | {chosen_test[key]:.3f} |")
card_lines += [
    "",
    f"PCK gain over stock on test, 95% bootstrap interval: {test_intervals.loc['2.5%', 'difference']:.3f} to "
    f"{test_intervals.loc['97.5%', 'difference']:.3f}",
    "",
    "## Known weaknesses",
    "",
    "- Trained and scored on COCO photos, not on SAMS cameras. Workers in hard hats, vests, and gloves are rare in COCO.",
    "- Wrists and ankles are the weakest joints, and they decide gloves and safety shoes.",
    "- In crops with several people, the service may keep a neighbour's pose.",
    "- See results/vision/pose/best_val_robustness.csv for the drop under blur, darkness, compression, and low resolution.",
]
(MODELS_DIR / "model_card.md").write_text("\n".join(card_lines) + "\n")
print("\n".join(card_lines))

## 12. Limitations and next steps

**Known limitations**

1. **COCO is not a construction site.** The crops are everyday photos: sports, streets, kitchens. Hard hats, high-visibility vests, gloves, and safety boots are rare, and so are industrial backgrounds and CCTV angles from above. The scores here say how well the model places joints on people in general, not on SAMS workers.
2. **Val 2017 only.** About 2,700 photos with labelled people are used, so the stock model can be scored fairly on unseen photos. Training on the 57,000 photos of train 2017 would give more data, but the stock model has already seen them, so its comparison would no longer be fair.
3. **Unlabelled is not invisible.** COCO marks a joint `v=0` when it was not annotated, which is usually because it is out of view, but not always. `false_joint_rate` therefore overstates mistakes a little.
4. **The service picks the pose, not the label.** `service_pck` follows the service and keeps the highest-scoring pose with a head or shoulder. In a crowded crop that may be a neighbour, which this score counts as an error on purpose, because it is one in SAMS too.
5. **Thresholds are borrowed.** The person threshold comes from the PPE model and the joint threshold is the service's fixed 0.3. Section 10.3 shows whether 0.3 suits this model, but the service is not changed by this notebook.

**Next steps**

1. **Label your own crops.** The SAMS archive (`monitoring/services/mlops.py` export) already saves frames. Label joints on a few hundred worker crops (for example with CVAT or Label Studio, exported to YOLO pose format), put them in `SITE_POSE_DIR`, and rerun. Even a small site set usually helps more than any setting here. Keep some of them aside as a site test set.
2. **Retune `_KP_MIN`** if section 10.3 shows the best balance far from 0.3.
3. **Retrain the distance network** (notebook 04) after promoting a new pose model, because its pose features (`pose_height_rel`, `shoulder_width_rel`, ...) were computed with the stock model's joints.